# Shared-bike demand forecasting around metro stations: analysis code

Code for *"Do deep spatiotemporal models improve on a calibrated statistical baseline for forecasting shared-bike demand around metro stations?"*.

Run Section 0 first. Sections 1-5 build the panels and the coupling diagnostics; Section 6 holds the model code shared by
Sections 7-9; Sections 10-11 repeat the comparison under the 1-km protocol; Section 12 draws the figures.
Every section starts with its own configuration cell and reads its inputs from disk, so sections can be run one at a time.
`OUT_DIR` is reset at the start of each section.

| Section | Output | Paper |
|---|---|---|
| 1 | daily merged bike/AFC files (500 m) | input of Sections 4 and 10 |
| 2 | station x time panels (50/100/200/500 m) | main panel |
| 3 | station-level Granger tests | Table 1 |
| 4-5 | distance-ring panel, PPML elasticities | Table 2 |
| 7 | LSTM / ST-GCN / MBC-GCN at station level | Tables 5, 7 and B.2 |
| 8 | statistical core, SC-LSTM hybrid, combination | Tables 6 and 9, Figure 10 |
| 9 | network-total forecasts | Tables 7 and 8 |
| 10-11 | 1-km panel, pipeline vs published scores | Table 10 |
| 12 | figures | Figures 7, 8, 9 and 11 |

## 0. Setup: paths

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()   # repository root
DATA = ROOT / "data"                                      # inputs (see README)
OUT  = ROOT / "outputs"                                   # everything written by this notebook
RAW_BIKE = DATA / "bike20221212-20230101.csv"             # lock/unlock log: BICYCLE_ID, DATA_TIME, LONGITUDE, LATITUDE, LOCK_STATUS
COORDS   = DATA / "metro_station_coordinates.csv"         # stationnames, lon, lat
INFO     = DATA / "station_info.csv"                      # FSTR_LINEID, FSTR_STATIONID, FSTR_STATIONNAME
AFC_DIR  = DATA / "afc_5min"                              # HIST_STO_5MFLOW_5M_YYYYMMDD.CSV

MERGED_DIR    = OUT / "merged" / "500m"
PANEL_DIR     = OUT / "panel"
RING_DIR      = OUT / "rings"
FORECAST_DIR  = OUT / "forecast"
PANEL_1KM_DIR = OUT / "panel_1km"
STATS_1KM_DIR = OUT / "stats_1km"
FIG_DIR       = OUT / "figures"
for d in (MERGED_DIR, PANEL_DIR, RING_DIR, FORECAST_DIR, PANEL_1KM_DIR, STATS_1KM_DIR, FIG_DIR): d.mkdir(parents=True, exist_ok=True)

## 1. Daily merged files: bike events matched to stations within 500 m, on the 5-min AFC grid

In [ ]:
# Functions: travel-chain reconstruction, station matching, aggregation to the AFC grid
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# WGS84 equatorial radius, used for the Web Mercator projection.
EARTH_RADIUS_M = 6378137.0
DEG = np.pi / 180.0

# Shanghai bounding box used to drop obviously corrupted GPS fixes.
LON_MIN, LON_MAX = 120.0, 122.0
LAT_MIN, LAT_MAX = 30.0, 32.0

# Segments shorter than this are treated as GPS jitter rather than trips.
MIN_SEGMENT_SECONDS = 10


def lonlat_to_mercator(lon_deg: np.ndarray, lat_deg: np.ndarray):
    """Project WGS84 degrees to Web Mercator metres (EPSG:3857)."""
    x = EARTH_RADIUS_M * lon_deg * DEG
    lat_rad = np.clip(lat_deg, -85.05112878, 85.05112878) * DEG
    y = EARTH_RADIUS_M * np.log(np.tan(np.pi / 4.0 + lat_rad / 2.0))
    return x, y


def load_static_tables(station_coords_path: Path, station_info_path: Path) -> pd.DataFrame:
    """Build the station reference table (ids, name, coordinates) shared by every day."""
    station_coords = pd.read_csv(station_coords_path, encoding="utf-8")
    station_coords = station_coords.drop_duplicates().reset_index(drop=True)

    station_info = pd.read_csv(station_info_path, encoding="gbk")
    station_info = station_info.rename(columns={"FSTR_STATIONNAME": "stationnames"})

    # Line and station ids are used as join keys against the AFC files, where
    # they may be stored with different dtypes or padding.
    for col in ("FSTR_LINEID", "FSTR_STATIONID"):
        station_info[col] = station_info[col].astype(str).str.strip()

    # One row per station with coordinates; the line name is not needed here.
    stations = (
        station_info.merge(station_coords, how="inner", on="stationnames")
        .drop(columns=["linename"], errors="ignore")
        .drop_duplicates()
        .reset_index(drop=True)
    )
    return stations


def reconstruct_travel_chain(bike_all: pd.DataFrame, day: datetime) -> pd.DataFrame:
    """Rebuild one day of continuous bicycle timelines.

    Lock-status events are asynchronous, so a mirrored copy of each record is
    generated one second earlier with the lock state inverted. Merging the two
    copies turns the point events into continuous state intervals. Midnight
    anchors are then added at both ends of each bicycle's day so that the first
    and last intervals are not truncated.
    """
    day_str = day.strftime("%Y/%m/%d")
    bikes = bike_all[bike_all["DATA_TIME"].astype(str).str.contains(day_str, na=False)].copy()

    bikes["DATA_TIME"] = pd.to_datetime(bikes["DATA_TIME"], errors="coerce")
    bikes = bikes.dropna(subset=["DATA_TIME"]).sort_values("DATA_TIME").reset_index(drop=True)

    # Replace the raw bicycle identifier with a compact integer id.
    bikes = bikes.sort_values("BICYCLE_ID").reset_index(drop=True)
    bike_ids = bikes["BICYCLE_ID"].drop_duplicates().to_frame().reset_index(drop=True)
    bike_ids["id"] = range(len(bike_ids))
    bikes = bikes.merge(bike_ids, on="BICYCLE_ID", how="left")

    # Some exports carry a mangled COMPANY_ID header; normalise it.
    for col in bikes.columns:
        if "COMPANY" in col.upper():
            bikes = bikes.rename(columns={col: "type"})
            break

    bikes = bikes[
        bikes["LONGITUDE"].between(LON_MIN, LON_MAX, inclusive="neither")
        & bikes["LATITUDE"].between(LAT_MIN, LAT_MAX, inclusive="neither")
    ].copy()

    # Mirrored records: one second earlier with the lock state reversed.
    mirrored = bikes.copy()
    mirrored["DATA_TIME"] = mirrored["DATA_TIME"] - timedelta(seconds=1)
    mirrored["LOCK_STATUS"] = 1 - mirrored["LOCK_STATUS"]

    events = pd.concat([mirrored, bikes], ignore_index=True)
    events["DATA_TIME"] = pd.to_datetime(events["DATA_TIME"], errors="coerce")
    events["DATE"] = events["DATA_TIME"].dt.date

    # Keep only bicycles with at least two valid records on that day.
    events = (
        events.groupby(["id", "DATE"]).filter(lambda g: len(g) > 1).reset_index(drop=True)
    )

    # Midnight anchors at both ends of the day.
    ordered = events.sort_values(["id", "DATA_TIME"])
    start = ordered.groupby("id").head(1).copy()
    start.loc[:, "DATA_TIME"] = pd.to_datetime(day.strftime("%Y-%m-%d 00:00:00"))
    end = ordered.groupby("id").tail(1).copy()
    end.loc[:, "DATA_TIME"] = pd.to_datetime(
        (day + timedelta(days=1)).strftime("%Y-%m-%d 00:00:00")
    )

    chain = pd.concat([events, start, end], ignore_index=True)
    chain["DATA_TIME"] = pd.to_datetime(chain["DATA_TIME"], errors="coerce")
    chain = (
        chain.dropna(subset=["DATA_TIME"])
        .sort_values(["id", "DATA_TIME"])
        .reset_index(drop=True)
    )

    # Attach the following record of the same bicycle to each row.
    grouped = chain.groupby("id")
    chain["id_next"] = grouped["id"].shift(-1)
    chain["DATA_TIME_next"] = grouped["DATA_TIME"].shift(-1)
    chain["LOCK_STATUS_next"] = grouped["LOCK_STATUS"].shift(-1)
    chain["LONGITUDE_next"] = grouped["LONGITUDE"].shift(-1)
    chain["LATITUDE_next"] = grouped["LATITUDE"].shift(-1)
    chain["dt_s"] = (chain["DATA_TIME_next"] - chain["DATA_TIME"]).dt.total_seconds()

    valid = chain["id"].eq(chain["id_next"]) & chain["dt_s"].gt(0)
    # Locked-to-locked spans are parking, unlocked-to-unlocked spans are riding.
    chain["stop_dt_s"] = chain["dt_s"].where(
        valid & chain["LOCK_STATUS"].eq(1) & chain["LOCK_STATUS_next"].eq(1)
    )
    chain["ride_dt_s"] = chain["dt_s"].where(
        valid & chain["LOCK_STATUS"].eq(0) & chain["LOCK_STATUS_next"].eq(0)
    )

    chain = chain[chain["id"] == chain["id_next"]].reset_index(drop=True)
    chain[["dt_s", "stop_dt_s", "ride_dt_s"]] = chain[
        ["dt_s", "stop_dt_s", "ride_dt_s"]
    ].fillna(0)
    return chain


def match_bikes_to_stations(
    chain: pd.DataFrame, stations: pd.DataFrame, buffer_m: float
) -> pd.DataFrame:
    """Pair bike events with every station within `buffer_m` Manhattan metres.

    Manhattan rather than Euclidean distance is used because it approximates
    walking distance in a dense gridded street network.
    """
    stations = stations.copy()
    bikes = chain.copy()

    stations["_xm"], stations["_ym"] = lonlat_to_mercator(
        stations["lon"].to_numpy(), stations["lat"].to_numpy()
    )
    bikes["_xm"], bikes["_ym"] = lonlat_to_mercator(
        bikes["LONGITUDE"].to_numpy(), bikes["LATITUDE"].to_numpy()
    )

    bx = bikes["_xm"].to_numpy()
    by = bikes["_ym"].to_numpy()

    cols = stations.columns.tolist()
    ix_xm, ix_ym = cols.index("_xm"), cols.index("_ym")

    matches = []
    for row in stations.to_numpy():
        xm, ym = row[ix_xm], row[ix_ym]
        mask = (np.abs(bx - xm) + np.abs(by - ym)) <= float(buffer_m)
        if not mask.any():
            continue
        hit = bikes.loc[mask].copy()
        for col, value in zip(cols, row):
            hit[f"station__{col}"] = value
        matches.append(hit)

    if not matches:
        return bikes.head(0).copy()

    pairs = pd.concat(matches, ignore_index=True)

    bx_m, by_m = lonlat_to_mercator(
        pairs["LONGITUDE"].to_numpy(), pairs["LATITUDE"].to_numpy()
    )
    pairs["distance MB"] = np.abs(bx_m - pairs["station___xm"].to_numpy()) + np.abs(
        by_m - pairs["station___ym"].to_numpy()
    )
    pairs = pairs.drop(columns=["_xm", "_ym", "station___xm", "station___ym"], errors="ignore")
    return pairs.sort_values(["id", "DATA_TIME"]).reset_index(drop=True)


def aggregate_to_afc_grid(pairs: pd.DataFrame, flow_5min: pd.DataFrame) -> pd.DataFrame:
    """Join the matched bike events onto the 5-minute AFC windows and count flows."""
    flow_5min = flow_5min.rename(
        columns={
            "FSTR_LINEID": "station__FSTR_LINEID",
            "FSTR_STATIONID": "station__FSTR_STATIONID",
        }
    )
    merged = flow_5min.merge(
        pairs, how="left", on=["station__FSTR_LINEID", "station__FSTR_STATIONID"]
    )
    merged = merged.sort_values(["station__stationnames", "FDT_STARTTIME"]).reset_index(
        drop=True
    )

    # Initial stock: bicycles already parked in the station area before the
    # first departure of the day. The frame is sorted by station and window, so
    # a grouped cumulative sum marks the rows preceding that first departure.
    station = merged["station__stationnames"]
    first_departure = merged["LOCK_STATUS"].eq(1) & merged["LOCK_STATUS_next"].eq(0)
    before_first = first_departure.groupby(station).cumsum().eq(0)
    parked = merged["LOCK_STATUS"].eq(1) & merged["LOCK_STATUS_next"].eq(1)
    merged["initial number"] = (
        (parked & before_first).astype(int).groupby(station).transform("sum")
    )

    for col in ["DATA_TIME", "DATA_TIME_next", "FDT_STARTTIME", "FDT_ENDTIME"]:
        merged[col] = pd.to_datetime(merged[col], errors="coerce")

    in_window = (merged["DATA_TIME"] >= merged["FDT_STARTTIME"]) & (
        merged["DATA_TIME"] <= merged["FDT_ENDTIME"]
    )
    merged = merged[in_window].reset_index(drop=True)

    merged["dt_s"] = (merged["DATA_TIME_next"] - merged["DATA_TIME"]).dt.total_seconds()
    in_window = (merged["DATA_TIME"] >= merged["FDT_STARTTIME"]) & (
        merged["DATA_TIME"] <= merged["FDT_ENDTIME"]
    )
    long_enough = merged["dt_s"] > MIN_SEGMENT_SECONDS

    # ride_in  = unlock -> lock  (a bicycle arrives at the station area)
    # ride_out = lock  -> unlock (a bicycle departs from the station area)
    merged["_ride_in_flag"] = (
        merged["LOCK_STATUS"].eq(0)
        & merged["LOCK_STATUS_next"].eq(1)
        & long_enough
        & in_window
    ).astype(int)
    merged["_ride_out_flag"] = (
        merged["LOCK_STATUS"].eq(1)
        & merged["LOCK_STATUS_next"].eq(0)
        & long_enough
        & in_window
    ).astype(int)

    keys = ["station__stationnames", "FDT_STARTTIME", "FDT_ENDTIME"]
    merged["ride_in"] = merged.groupby(keys)["_ride_in_flag"].transform("sum")
    merged["ride_out"] = merged.groupby(keys)["_ride_out_flag"].transform("sum")
    merged["delta_bikes"] = merged["ride_in"] - merged["ride_out"]
    return merged.drop(columns=["_ride_in_flag", "_ride_out_flag"])


def process_one_day(
    day: datetime,
    bike_all: pd.DataFrame,
    stations: pd.DataFrame,
    flow_dir: Path,
    out_root: Path,
    buffers: list[float],
) -> None:
    """Run the full pipeline for one day and write one CSV per buffer radius."""
    day_tag = day.strftime("%Y%m%d")

    flow_path = flow_dir / f"HIST_STO_5MFLOW_5M_{day_tag}.CSV"
    if not flow_path.exists():
        print(f"[skip] missing AFC file: {flow_path}")
        return

    flow_5min = pd.read_csv(flow_path, encoding="utf-8")
    for col in ("FSTR_LINEID", "FSTR_STATIONID"):
        flow_5min[col] = flow_5min[col].astype(str).str.strip()

    chain = reconstruct_travel_chain(bike_all, day)

    for buffer_m in buffers:
        pairs = match_bikes_to_stations(chain, stations, buffer_m)
        panel = aggregate_to_afc_grid(pairs, flow_5min.copy())

        out_dir = out_root / f"{int(buffer_m)}m"
        out_dir.mkdir(parents=True, exist_ok=True)
        panel.to_csv(out_dir / f"merged_{day_tag}.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Run: one merged file per day
stations_tbl = load_static_tables(COORDS, INFO)
bike_all = pd.read_csv(RAW_BIKE)
for day in tqdm(pd.date_range("2022-12-12", "2022-12-31", freq="D"), desc="days"):
    process_one_day(day=day, bike_all=bike_all, stations=stations_tbl, flow_dir=AFC_DIR, out_root=OUT / "merged", buffers=[500.0])
del bike_all

## 2. Main panel: bike trips matched to station catchments, merged with AFC flows

In [ ]:
# Configuration
import warnings, gc, time
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

BIKE_CSV, FLOW_DIR = RAW_BIKE, AFC_DIR
OUT_DIR = PANEL_DIR

DATES    = [f"202212{d:02d}" for d in range(12, 32)]
BUFFERS  = [50, 100, 200, 500]          # Manhattan metres
FREQS    = {"5min": "5min", "1h": "h"}

MIN_MIN, MAX_MIN, MIN_M, MAX_M = 1, 120, 100, 10000

In [ ]:
# Functions: trip extraction, catchment matching, AFC grid, panels
from scipy.spatial import cKDTree

R = 6378137.0; DEG = np.pi/180
def to_merc(lon, lat):
    lon = np.asarray(lon, float); lat = np.clip(np.asarray(lat, float), -85.05, 85.05)
    return R*lon*DEG, R*np.log(np.tan(np.pi/4 + lat*DEG/2))

def norm_key(s: pd.Series) -> pd.Series:
    """'09' / '0928' / '928.0' / ' 928 ' -> '9' / '928' / '928' / '928' so AFC and station_info keys agree."""
    s = s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    num = s.str.fullmatch(r"\d+")
    return s.where(~num, s.str.lstrip("0").replace("", "0"))

def load_stations(coords_path, info_path):
    coords = pd.read_csv(coords_path, encoding="utf-8").drop_duplicates()
    info = pd.read_csv(info_path, encoding="gbk").rename(columns={"FSTR_STATIONNAME": "stationnames"})
    for c in ("FSTR_LINEID", "FSTR_STATIONID"):
        info[c] = norm_key(info[c])
    st = info.merge(coords, on="stationnames", how="inner").drop(columns=["linename"], errors="ignore").drop_duplicates()
    # one physical station = one name/coords; keep list of (line, stationid) for AFC join
    nodes = st.groupby("stationnames", as_index=False).agg(lon=("lon", "first"), lat=("lat", "first"))
    nodes["node"] = range(len(nodes))
    keys = st[["FSTR_LINEID", "FSTR_STATIONID", "stationnames"]].drop_duplicates().merge(nodes[["stationnames", "node"]], on="stationnames")
    return nodes, keys

def extract_trips(bikes: pd.DataFrame, min_min=1, max_min=120, min_m=100, max_m=10000):
    """unlock (0) followed by lock (1) on the same bike = one trip."""
    b = bikes.sort_values(["BICYCLE_ID", "DATA_TIME"]).reset_index(drop=True)
    # collapse runs of identical status (GPS pings while riding / repeated lock reports):
    # keep the FIRST record of each run, so a trip starts at the first unlock and ends at the first lock
    prev = b.groupby("BICYCLE_ID", sort=False)["LOCK_STATUS"].shift()
    b = b[prev.isna() | (b["LOCK_STATUS"] != prev)].reset_index(drop=True)
    g = b.groupby("BICYCLE_ID", sort=False)
    for c in ["DATA_TIME", "LONGITUDE", "LATITUDE", "LOCK_STATUS"]:
        b[c + "_next"] = g[c].shift(-1)
    t = b[(b["LOCK_STATUS"] == 0) & (b["LOCK_STATUS_next"] == 1)].copy()
    t["dur_min"] = (t["DATA_TIME_next"] - t["DATA_TIME"]).dt.total_seconds() / 60
    x0, y0 = to_merc(t["LONGITUDE"], t["LATITUDE"]); x1, y1 = to_merc(t["LONGITUDE_next"], t["LATITUDE_next"])
    t["dist_m"] = np.hypot(x1 - x0, y1 - y0)
    t = t[t["dur_min"].between(min_min, max_min) & t["dist_m"].between(min_m, max_m)]
    return t.rename(columns={"DATA_TIME": "stime", "LONGITUDE": "slon", "LATITUDE": "slat",
                             "DATA_TIME_next": "etime", "LONGITUDE_next": "elon", "LATITUDE_next": "elat"}
                    )[["BICYCLE_ID", "stime", "slon", "slat", "etime", "elon", "elat", "dur_min", "dist_m"]].reset_index(drop=True)

def match_points(lon, lat, nodes, max_r=500):
    """all (point, node) pairs within Manhattan distance max_r; returns point index, node, distance."""
    sx, sy = to_merc(nodes["lon"], nodes["lat"]); tree = cKDTree(np.c_[sx, sy])
    px, py = to_merc(lon, lat)
    hits = tree.query_ball_point(np.c_[px, py], r=max_r, p=1)
    pi = np.repeat(np.arange(len(px)), [len(h) for h in hits]); ni = np.concatenate([np.asarray(h, int) for h in hits]) if len(pi) else np.array([], int)
    d = np.abs(px[pi] - sx[ni]) + np.abs(py[pi] - sy[ni])
    return pd.DataFrame({"pt": pi, "node": nodes["node"].to_numpy()[ni], "dist": d})

def bike_counts(trips, nodes, buffers, freq):
    """per node × window: ride_out (trips starting within buffer, by stime), ride_in (ending, by etime)."""
    out = {}
    dep = match_points(trips["slon"], trips["slat"], nodes, max(buffers)); dep["t"] = trips["stime"].to_numpy()[dep["pt"]]
    arr = match_points(trips["elon"], trips["elat"], nodes, max(buffers)); arr["t"] = trips["etime"].to_numpy()[arr["pt"]]
    for r in buffers:
        d = dep[dep["dist"] <= r].assign(window=lambda x: x["t"].dt.floor(freq)).groupby(["node", "window"]).size().rename("ride_out")
        a = arr[arr["dist"] <= r].assign(window=lambda x: x["t"].dt.floor(freq)).groupby(["node", "window"]).size().rename("ride_in")
        out[r] = pd.concat([d, a], axis=1).fillna(0).reset_index()
    return out

def load_afc(flow_dir, dates, keys):
    frames = []
    for d in dates:
        p = Path(flow_dir) / f"HIST_STO_5MFLOW_5M_{d}.CSV"
        if not p.exists(): print("[warn] missing", p.name); continue
        f = pd.read_csv(p, usecols=["FSTR_LINEID", "FSTR_STATIONID", "FDT_STARTTIME", "FINT_IN", "FINT_OUT"], dtype=str)
        frames.append(f)
    afc = pd.concat(frames, ignore_index=True)
    for c in ("FSTR_LINEID", "FSTR_STATIONID"): afc[c] = norm_key(afc[c])
    afc["window"] = pd.to_datetime(afc["FDT_STARTTIME"], errors="coerce", format="mixed")
    for c in ("FINT_IN", "FINT_OUT"): afc[c] = pd.to_numeric(afc[c], errors="coerce").fillna(0)
    afc = afc.dropna(subset=["window"]).drop_duplicates(subset=["FSTR_LINEID", "FSTR_STATIONID", "window"])
    afc = afc.merge(keys, on=["FSTR_LINEID", "FSTR_STATIONID"], how="inner")
    # one physical station: sum over lines sharing the station
    return afc.groupby(["node", "window"], as_index=False)[["FINT_IN", "FINT_OUT"]].sum()

def build_panel(afc5, trips, nodes, buffers, freq):
    m = afc5.assign(window=afc5["window"].dt.floor(freq)).groupby(["node", "window"], as_index=False)[["FINT_IN", "FINT_OUT"]].sum()
    bc = bike_counts(trips, nodes, buffers, freq)
    panels = {}
    for r in buffers:
        p = m.merge(bc[r], on=["node", "window"], how="left")
        p[["ride_in", "ride_out"]] = p[["ride_in", "ride_out"]].fillna(0).astype(int)
        p = p.merge(nodes[["node", "stationnames", "lon", "lat"]], on="node")
        panels[r] = p.sort_values(["node", "window"]).reset_index(drop=True)
    return panels

In [ ]:
# Stations
nodes, keys = load_stations(COORDS, INFO)
print(f"{len(nodes)} physical stations (by name/coords), {len(keys)} (line, station) AFC keys")

In [ ]:
# Bike log -> trips
t0 = time.time()
bikes = pd.read_csv(BIKE_CSV, usecols=lambda c: c.upper().endswith(("BICYCLE_ID","LONGITUDE","LATITUDE","LOCK_STATUS","DATA_TIME")))
bikes.columns = [c if not c.upper().endswith("COMPANY_ID") else "type" for c in bikes.columns]
bikes["DATA_TIME"] = pd.to_datetime(bikes["DATA_TIME"], errors="coerce", format="mixed")
bikes = bikes.dropna(subset=["DATA_TIME"])
bikes = bikes[bikes["LONGITUDE"].between(120, 122) & bikes["LATITUDE"].between(30, 32)]
bikes["LOCK_STATUS"] = pd.to_numeric(bikes["LOCK_STATUS"], errors="coerce")
bikes = bikes.dropna(subset=["LOCK_STATUS"])
print(f"{len(bikes):,} records, {bikes['BICYCLE_ID'].nunique():,} bikes, "
      f"{bikes['DATA_TIME'].min()} → {bikes['DATA_TIME'].max()}  ({time.time()-t0:.0f}s)")

trips = extract_trips(bikes, MIN_MIN, MAX_MIN, MIN_M, MAX_M)
del bikes; gc.collect()
trips = trips[trips["stime"].dt.strftime("%Y%m%d").isin(DATES)].reset_index(drop=True)
print(f"{len(trips):,} trips after cleaning  ({time.time()-t0:.0f}s)")
print(trips["dur_min"].describe(percentiles=[.25,.5,.75,.9]).round(1))
trips.to_parquet(OUT_DIR / "trips.parquet", index=False)

In [ ]:
# AFC grid and panels (one CSV per resolution and catchment)
afc5 = load_afc(FLOW_DIR, DATES, keys)
print(f"AFC 5-min grid: {len(afc5):,} station-windows, {afc5['node'].nunique()} stations")

panels = {}
for name, freq in FREQS.items():
    panels[name] = build_panel(afc5, trips, nodes, BUFFERS, freq)
    for r, p in panels[name].items():
        p.to_csv(OUT_DIR / f"panel_{name}_{r}m.csv", index=False, encoding="utf-8-sig")
    p = panels[name][100]
    print(f"[{name}] 100 m: {p['node'].nunique()} stations x {p['window'].nunique()} windows = {len(p):,} rows | "
          f"zero share ride_out {(p['ride_out']==0).mean():.1%}, ride_in {(p['ride_in']==0).mean():.1%} | "
          f"mean ride_out {p['ride_out'].mean():.2f}, ride_in {p['ride_in'].mean():.2f}, FINT_OUT {p['FINT_OUT'].mean():.1f}")

## 3. Station-level Granger tests (Table 1)

Hourly series, log(1+y), first-differenced; smallest F-test p-value over lags of 1-3 hours.

In [ ]:
# Functions: deseasonalisation, BIC lag order, Granger F-tests
import numpy as np, pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
warnings.filterwarnings("ignore")

def service_hours(panel, start="05:30", end="23:30"):
    h = panel["window"].dt.time
    return panel[(h >= pd.to_datetime(start).time()) & (h <= pd.to_datetime(end).time())]

def _granger_p(y, x, maxlag):
    """p-value (ssr F-test) that x Granger-causes y, min over lags 1..maxlag, and best lag."""
    d = pd.DataFrame({"y": y, "x": x}).dropna()
    if len(d) < 10 * maxlag or d["y"].std() == 0 or d["x"].std() == 0:
        return np.nan, np.nan
    try:
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):          # older statsmodels prints; newer has no verbose kw
            res = grangercausalitytests(d[["y", "x"]], maxlag=maxlag)
        ps = [res[l][0]["ssr_ftest"][1] for l in range(1, maxlag + 1)]
        return float(min(ps)), int(np.argmin(ps)) + 1
    except Exception:
        return np.nan, np.nan

GRANGER_COLS = ["node", "stationnames", "n", "mean_ride_out", "mean_ride_in",
                "exit->dep_p", "exit->dep_lag", "dep->exit_p", "dep->exit_lag",
                "arr->entry_p", "arr->entry_lag", "entry->arr_p", "entry->arr_lag"]

def granger_by_station(panel, maxlag=4, min_mean_bike=0.5, log1p=True, diff=True):
    """Per-station Granger tests in both directions for the two coupled pairs.
    Series are log1p-transformed and first-differenced, restricted to service hours."""
    rows = []
    p = service_hours(panel)
    for node, g in p.groupby("node"):
        if g["ride_out"].mean() < min_mean_bike or g["ride_in"].mean() < min_mean_bike:
            continue
        g = g.sort_values("window")
        s = {c: (np.log1p(g[c]) if log1p else g[c].astype(float)) for c in ["FINT_IN", "FINT_OUT", "ride_in", "ride_out"]}
        if diff:
            s = {c: v.diff().dropna() for c, v in s.items()}
        r = {"node": node, "stationnames": g["stationnames"].iloc[0], "n": len(g),
             "mean_ride_out": g["ride_out"].mean(), "mean_ride_in": g["ride_in"].mean()}
        r["exit->dep_p"], r["exit->dep_lag"] = _granger_p(s["ride_out"], s["FINT_OUT"], maxlag)   # metro exits -> bike departures
        r["dep->exit_p"], r["dep->exit_lag"] = _granger_p(s["FINT_OUT"], s["ride_out"], maxlag)
        r["arr->entry_p"], r["arr->entry_lag"] = _granger_p(s["FINT_IN"], s["ride_in"], maxlag)  # bike arrivals -> metro entries
        r["entry->arr_p"], r["entry->arr_lag"] = _granger_p(s["ride_in"], s["FINT_IN"], maxlag)
        rows.append(r)
    return pd.DataFrame(rows, columns=GRANGER_COLS)

def granger_summary(res, alpha=0.05):
    if res.empty:
        return pd.DataFrame([{"direction": d, "stations_tested": 0} for d in
                             ["exit->dep", "dep->exit", "arr->entry", "entry->arr"]])
    out = []
    for d in ["exit->dep", "dep->exit", "arr->entry", "entry->arr"]:
        p = res[f"{d}_p"].dropna(); lag = res.loc[res[f"{d}_p"] < alpha, f"{d}_lag"]
        out.append({"direction": d, "stations_tested": len(p),
                    f"share_p<{alpha}": (p < alpha).mean(),
                    "median_p": p.median(),
                    "best_lag_mode": lag.mode().iloc[0] if len(lag) else np.nan,
                    "best_lag_dist": lag.value_counts(normalize=True).sort_index().round(2).to_dict()})
    return pd.DataFrame(out)

In [ ]:
# Run: tests by catchment; the summary is Table 1
GRANGER_RES = "1h"
BUFFERS, OUT_DIR = [50, 100, 200, 500], PANEL_DIR
panels = {GRANGER_RES: {r: pd.read_csv(PANEL_DIR / f"panel_{GRANGER_RES}_{r}m.csv", parse_dates=["window"]) for r in BUFFERS}}
MAXLAG      = 3

granger_tables, granger_station = {}, {}
for r in BUFFERS:
    p = panels[GRANGER_RES][r]
    # station must average >= 0.2 rides/window and >= half of the buffer-wide station mean
    thr = max(0.2, 0.5 * p.groupby("node")["ride_out"].mean().mean())
    res = granger_by_station(p, maxlag=MAXLAG, min_mean_bike=thr)
    print(f"{r} m: threshold {thr:.2f} rides/window -> {len(res)} of {p['node'].nunique()} stations tested")
    granger_station[r] = res
    s = granger_summary(res); s.insert(0, "buffer_m", r)
    granger_tables[r] = s
    res.to_csv(OUT_DIR / f"granger_station_{GRANGER_RES}_{r}m.csv", index=False)

granger_tab = pd.concat(granger_tables.values(), ignore_index=True)
granger_tab.to_csv(OUT_DIR / f"granger_summary_{GRANGER_RES}.csv", index=False)
print(granger_tab.drop(columns="best_lag_dist", errors="ignore").to_string(index=False))
print()
for r in BUFFERS:
    if "best_lag_dist" in granger_tables[r]:
        print(f"{r} m best-lag distribution:", granger_tables[r].set_index("direction")["best_lag_dist"].to_dict())

## 4. Distance-ring panel: nearest station, 50 m rings, 5-min slots

In [ ]:
# Imports
import re, warnings
from pathlib import Path
import numpy as np, pandas as pd
import pyarrow.csv as pcsv
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

In [ ]:
# Configuration
FILES      = sorted(MERGED_DIR.glob("merged_2022*.csv"))
OUT_ROOT   = RING_DIR

EDGES      = [0, 50, 100, 150, 200, 300, 400, 500]
EVENT_DEFS = ["raw"]
MIN_SEGMENT_SECONDS = 10

print(len(FILES), "daily files:", FILES[0].name, "...", FILES[-1].name)

In [ ]:
# Functions: read a daily file, count lock/unlock events by ring
USECOLS = ["station__FSTR_STATIONID", "station__stationnames", "FDT_STARTTIME", "FDT_ENDTIME",
           "FINT_IN", "FINT_OUT", "BICYCLE_ID", "DATA_TIME", "LOCK_STATUS", "LOCK_STATUS_next",
           "dt_s", "LONGITUDE", "LATITUDE", "LONGITUDE_next", "LATITUDE_next", "distance MB"]

def read_merged(path):
    t = pcsv.read_csv(path, convert_options=pcsv.ConvertOptions(include_columns=USECOLS),
                      read_options=pcsv.ReadOptions(block_size=256 << 20)).to_pandas()
    for c in ("FDT_STARTTIME", "FDT_ENDTIME", "DATA_TIME"):
        t[c] = pd.to_datetime(t[c])
    t["station__FSTR_STATIONID"] = t["station__FSTR_STATIONID"].astype(str).str.strip()
    return t

def process_file(path):
    t = read_merged(path)
    grid = (t.drop_duplicates(["station__FSTR_STATIONID", "FDT_STARTTIME"])
              .groupby(["station__stationnames", "FDT_STARTTIME", "FDT_ENDTIME"])
              .agg(n_lines=("station__FSTR_STATIONID", "nunique"), FINT_IN=("FINT_IN", "sum"), FINT_OUT=("FINT_OUT", "sum"))
              .reset_index())
    same_loc = np.isclose(t["LONGITUDE"], t["LONGITUDE_next"]) & np.isclose(t["LATITUDE"], t["LATITUDE_next"])
    mirror = (t["dt_s"] == 1) & (t["LOCK_STATUS_next"] == 1 - t["LOCK_STATUS"]) & same_loc
    t["in_raw"]  = (~mirror & t["LOCK_STATUS"].eq(1)).astype(int)
    t["out_raw"] = (~mirror & t["LOCK_STATUS"].eq(0)).astype(int)
    t["_m"] = mirror.astype(int)
    t = t.sort_values("distance MB").drop_duplicates(["BICYCLE_ID", "DATA_TIME", "LOCK_STATUS", "_m"])
    ev = t[(t["in_raw"] + t["out_raw"]) > 0].copy()
    ev["FDT_STARTTIME"] = ev["DATA_TIME"].dt.floor("5min")
    ev["ring"] = pd.cut(ev["distance MB"], bins=EDGES, right=False, include_lowest=True, labels=False)
    ev = ev.dropna(subset=["ring"]); ev["ring"] = ev["ring"].astype(int)
    out = {}
    for ed in EVENT_DEFS:
        cnt = ev.groupby(["station__stationnames", "FDT_STARTTIME", "ring"])[[f"in_{ed}", f"out_{ed}"]].sum().unstack("ring", fill_value=0)
        cnt.columns = [f"ride_{c.split('_')[0]}_r{EDGES[b]}_{EDGES[b+1]}" for c, b in cnt.columns]
        cnt = cnt.reset_index()
        panel = grid.merge(cnt, on=["station__stationnames", "FDT_STARTTIME"], how="left")
        ring_cols = [f"ride_{k}_r{EDGES[b]}_{EDGES[b+1]}" for b in range(len(EDGES) - 1) for k in ("in", "out")]
        for c in ring_cols:
            if c not in panel: panel[c] = 0
        panel[ring_cols] = panel[ring_cols].fillna(0).astype(int)
        out[ed] = panel.rename(columns={"station__stationnames": "stationnames"})[
            ["stationnames", "FDT_STARTTIME", "FDT_ENDTIME", "n_lines", "FINT_IN", "FINT_OUT"] + ring_cols]
    stats = dict(rows=len(t), mirror_pct=100 * mirror.mean(), raw_lock=int(t.in_raw.sum()), raw_unlock=int(t.out_raw.sum()))
    return out, stats

In [ ]:
# Run: ring panel for all days
parts = {ed: [] for ed in EVENT_DEFS}; log = []
for f in tqdm(FILES, desc="days"):
    out, st = process_file(f)
    for ed in EVENT_DEFS: parts[ed].append(out[ed])
    log.append({"file": f.name, **st})
log = pd.DataFrame(log); print(log.to_string(index=False))

for ed in EVENT_DEFS:
    panel = pd.concat(parts[ed], ignore_index=True).sort_values(["stationnames", "FDT_STARTTIME"])
    p = OUT_ROOT / f"station_ring_panel_5min_{ed}.csv"
    panel.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"{ed:>8}: {len(panel):,} station-windows, {panel.stationnames.nunique()} stations -> {p}")
(OUT_ROOT / "ring_edges.txt").write_text(" ".join(map(str, EDGES)))

## 5. PPML elasticities by distance ring and cumulative disk (Table 2)

In [ ]:
# Configuration
import re, warnings
from pathlib import Path
import numpy as np, pandas as pd, pyfixest as pf
warnings.filterwarnings("ignore")

EVENT_DEFS = ["raw"]
AGG_LIST   = [60]
ADD_LAG    = True
MIN_NONZERO = 2.0
OUT_DIR    = RING_DIR

In [ ]:
# Functions: aggregation, PPML fit, one setting
def build_panel(df, ring_cols, edges, agg):
    rings = list(zip(edges[:-1], edges[1:]))
    p = df.copy()
    if agg != 5:
        p["t"] = p["FDT_STARTTIME"].dt.floor(f"{agg}min")
        g = p.groupby(["stationnames", "t"])
        p = g[["FINT_IN", "FINT_OUT"] + ring_cols].sum().join(g.size().rename("n")).reset_index()
        p = p[p["n"] == agg // 5].drop(columns="n").rename(columns={"t": "FDT_STARTTIME"})
    for k in ("IN", "OUT"):
        p[f"lFINT_{k}"] = np.log1p(p[f"FINT_{k}"])
    if ADD_LAG:
        prev = p[["stationnames", "FDT_STARTTIME", "lFINT_IN", "lFINT_OUT"]].copy()
        prev["FDT_STARTTIME"] += pd.Timedelta(minutes=agg)
        prev = prev.rename(columns={"lFINT_IN": "lFINT_IN_lag1", "lFINT_OUT": "lFINT_OUT_lag1"})
        p = p.merge(prev, on=["stationnames", "FDT_STARTTIME"], how="left").dropna(subset=["lFINT_IN_lag1"])
    p["st"]   = p["stationnames"].astype(str)
    p["tod"]  = (p["FDT_STARTTIME"].dt.hour * 60 + p["FDT_STARTTIME"].dt.minute) // agg
    p["date"] = p["FDT_STARTTIME"].dt.strftime("%Y-%m-%d")
    for hi in edges[1:]:
        for k in ("in", "out"):
            p[f"disk_{k}_{hi}"] = p[[f"ride_{k}_r{lo}_{h}" for lo, h in rings if h <= hi]].sum(axis=1)
    return p

def fit(p, y, k):
    rhs = f"lFINT_{k}" + (f" + lFINT_{k}_lag1" if ADD_LAG else "")
    m = pf.fepois(f"{y} ~ {rhs} | st + tod + date", data=p, vcov={"CRV1": "st"})
    t = m.tidy().loc[f"lFINT_{k}"]
    return dict(beta=t["Estimate"], lo=t["2.5%"], hi=t["97.5%"], p=t["Pr(>|t|)"],
                nonzero_pct=100 * (p[y] > 0).mean(), N=len(p))

def run_setting(ed, agg):
    df = pd.read_csv(RING_DIR / f"station_ring_panel_5min_{ed}.csv", parse_dates=["FDT_STARTTIME"])
    ring_cols = [c for c in df.columns if re.match(r"ride_(in|out)_r\d+_\d+$", c)]
    edges = sorted({int(x) for c in ring_cols for x in re.findall(r"r(\d+)_(\d+)", c)[0]})
    rings = list(zip(edges[:-1], edges[1:]))
    p = build_panel(df, ring_cols, edges, agg)
    rows = []
    for lo, hi in rings:
        for k in ("IN", "OUT"):
            try: rows.append(dict(event=ed, agg=agg, kind="ring", range=f"{lo}-{hi}", mid=(lo + hi) / 2, Pair=k, **fit(p, f"ride_{k.lower()}_r{lo}_{hi}", k)))
            except Exception as e: print(ed, agg, "ring", lo, hi, k, "failed:", e)
    for hi in edges[1:]:
        for k in ("IN", "OUT"):
            try: rows.append(dict(event=ed, agg=agg, kind="disk", range=f"0-{hi}", mid=hi, Pair=k, **fit(p, f"disk_{k.lower()}_{hi}", k)))
            except Exception as e: print(ed, agg, "disk", hi, k, "failed:", e)
    print(f"{ed:>8} {agg:>2} min: sample {len(p):,} rows")
    return pd.DataFrame(rows), edges

In [ ]:
# Run
all_res = []
for ed in EVENT_DEFS:
    for agg in AGG_LIST:
        r, edges = run_setting(ed, agg)
        all_res.append(r)
res = pd.concat(all_res, ignore_index=True)
res.to_csv(OUT_DIR / "ring_search_all_settings.csv", index=False)

In [ ]:
# Summary: best disk, peak ring, decay edge
summary = []
for (ed, agg, k), s in res.groupby(["event", "agg", "Pair"]):
    disk = s[s.kind == "disk"].sort_values("mid")
    ring = s[(s.kind == "ring") & (s.nonzero_pct >= MIN_NONZERO)].sort_values("mid").reset_index(drop=True)
    best = disk.loc[disk["beta"].idxmax()]
    tie = disk[disk["hi"] >= best["lo"]]["mid"].tolist()
    ipk = ring["beta"].idxmax(); peak = ring.loc[ipk, "beta"]
    after = ring.loc[ipk + 1:]; dec = after[after["hi"] < peak / 2]
    edge = int(dec["range"].iloc[0].split("-")[0]) if len(dec) else np.nan
    summary.append(dict(event=ed, agg_min=agg, Pair=k, N=int(disk["N"].iloc[0]),
                        best_disk_m=int(best["mid"]), best_disk_beta=round(best["beta"], 3),
                        disks_within_CI=tie, ring_peak=ring.loc[ipk, "range"], decay_edge_m=edge))
summary = pd.DataFrame(summary)
print(summary.to_string(index=False))
summary.to_csv(OUT_DIR / "ring_search_summary.csv", index=False)

## 6. Model code shared by Sections 7-9: data tensors, folds, LSTM / ST-GCN / MBC-GCN, training, metrics

In [ ]:
# Shared model code
import math, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.distance import cdist
warnings.filterwarnings("ignore")

FLOW_BIKE  = ["ride_in", "ride_out"]
FLOW_METRO = ["FINT_IN", "FINT_OUT"]
TIME_FEATS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "day_level"]   # day_level = previous-day network log-volume (handles the December level shift)

def get_device():
    if torch.backends.mps.is_available(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)

# --------------------------------------------------------------------------
# panel -> (T, N, F) tensor
# --------------------------------------------------------------------------
def load_panel(path: Path, service=("05:30", "23:30")):
    p = pd.read_csv(path)
    p["window"] = pd.to_datetime(p["window"])
    h = p["window"].dt.time
    p = p[(h >= pd.to_datetime(service[0]).time()) & (h <= pd.to_datetime(service[1]).time())]
    return p.sort_values(["window", "node"]).reset_index(drop=True)

def build_graph(nodes: pd.DataFrame, threshold_km=2.0, sigma_km=1.0):
    R = 6378137.0; lon = nodes["lon"].to_numpy(float); lat = nodes["lat"].to_numpy(float)
    x = R*np.radians(lon); y = R*np.log(np.tan(np.pi/4 + np.radians(lat)/2))       # Web Mercator, same as the panel builder
    d = cdist(np.c_[x, y], np.c_[x, y]) / 1000.0
    a = np.exp(-(d ** 2) / (2 * sigma_km ** 2)); a[d > threshold_km] = 0.0
    a_t = a + np.eye(len(a)); dinv = np.diag(1 / np.sqrt(a_t.sum(1)))
    return torch.FloatTensor(dinv @ a_t @ dinv)

def panel_to_tensor(p: pd.DataFrame):
    """Returns tensor (T, N, F) with F = [ride_in, ride_out, FINT_IN, FINT_OUT, time feats...], plus windows, nodes."""
    nodes = p.drop_duplicates("node")[["node", "stationnames", "lon", "lat"]].sort_values("node").reset_index(drop=True)
    windows = np.sort(p["window"].unique())
    T, N = len(windows), len(nodes)
    wi = pd.Series(np.arange(T), index=windows); ni = pd.Series(np.arange(N), index=nodes["node"])
    X = np.zeros((T, N, 4), dtype=np.float32)
    X[wi[p["window"]].to_numpy(), ni[p["node"]].to_numpy(), :] = p[FLOW_BIKE + FLOW_METRO].to_numpy(np.float32)
    w = pd.DatetimeIndex(windows)
    hod = w.hour + w.minute / 60; dow = w.dayofweek
    tfe = np.stack([np.sin(2*np.pi*hod/24), np.cos(2*np.pi*hod/24), np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], 1).astype(np.float32)
    # previous-day network bike volume (log), broadcast to every window of the day; first day uses its own
    daily = pd.Series(X[:, :, :2].sum((1, 2)), index=w).groupby(w.date).sum()
    prev = daily.shift(1).fillna(daily.iloc[0]); lvl = np.log1p(prev.reindex(w.date).to_numpy()).astype(np.float32)
    lvl = (lvl - lvl.mean()) / (lvl.std() + 1e-6)
    tfe = np.c_[tfe, lvl]
    X = np.concatenate([X, np.repeat(tfe[:, None, :], N, 1)], 2)
    return X, w, nodes

# --------------------------------------------------------------------------
# rolling-origin folds on whole days
# --------------------------------------------------------------------------
def day_folds(windows: pd.DatetimeIndex, val_days=2, test_days=2, n_folds=3):
    days = sorted(set(windows.date)); nd = len(days)
    folds = []
    for k in range(n_folds):
        te_end = nd - k * test_days; te_start = te_end - test_days; va_start = te_start - val_days
        folds.append({"train": days[:va_start], "val": days[va_start:te_start], "test": days[te_start:te_end]})
    return folds[::-1]     # chronological order

def split_index(windows, fold):
    d = windows.date
    return {k: np.where(np.isin(d, v))[0] for k, v in fold.items()}

# --------------------------------------------------------------------------
# scaling: log1p flows, per-channel z-score fitted on train windows only
# --------------------------------------------------------------------------
def sanitize(X):
    """counts must be finite and >= 0 before log1p; time feats finite."""
    Y = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    Y[:, :, :4] = np.clip(Y[:, :, :4], 0, None); return Y

class Scaler:
    def fit(self, X, idx):
        z = np.log1p(sanitize(X)[idx, :, :4]); self.mu = z.mean((0, 1)); self.sd = z.std((0, 1)) + 1e-6; return self
    def transform(self, X):
        Y = sanitize(X); Y[:, :, :4] = (np.log1p(Y[:, :, :4]) - self.mu) / self.sd; return Y
    def inverse_targets(self, Z):          # Z: (..., 2) scaled bike targets -> counts
        return np.expm1(Z * self.sd[:2] + self.mu[:2]).clip(0)

class SeqDataset(Dataset):
    """x: window of `seq_len` steps ending at t (all features); y: bike flows at t+1 (scaled)."""
    def __init__(self, Xs, idx, seq_len, feat_cols):
        self.X = Xs; self.seq = seq_len; self.fc = feat_cols
        sidx = set(int(i) for i in idx)
        # target t+1 must belong to the split; the input window may reach back into earlier data (warm-up)
        self.t = [int(t) for t in idx if t - seq_len + 1 >= 0 and t + 1 < len(Xs) and (t + 1) in sidx]
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        t = self.t[i]
        x = torch.from_numpy(self.X[t - self.seq + 1: t + 1][:, :, self.fc])
        y = torch.from_numpy(self.X[t + 1][:, :2])
        return x, y

def make_loaders(Xs, splits, seq_len, feat_cols, batch):
    def ds(idx): return SeqDataset(Xs, idx, seq_len, feat_cols)
    return (DataLoader(ds(splits["train"]), batch, shuffle=True),
            DataLoader(ds(splits["val"]), batch), DataLoader(ds(splits["test"]), batch))

# --------------------------------------------------------------------------
# models
# --------------------------------------------------------------------------
class ChebGraphConv(nn.Module):
    def __init__(self, cin, cout, k, gain=1.0):
        super().__init__(); self.k = k
        self.w = nn.Parameter(torch.empty(k, cin, cout)); self.b = nn.Parameter(torch.zeros(cout))
        nn.init.xavier_uniform_(self.w, gain=gain)
    def forward(self, x, L):                       # x (B,T,N,C)
        B, T, N, C = x.shape; xf = x.reshape(B*T, N, C).contiguous()
        Lb = L.unsqueeze(0).expand(B*T, -1, -1).contiguous()   # explicit batch: MPS mis-handles broadcast matmul
        tp = xf; out = tp @ self.w[0]
        if self.k > 1:
            tc = torch.bmm(Lb, xf); out = out + tc @ self.w[1]
            for o in range(2, self.k):
                tn = 2*torch.bmm(Lb, tc) - tp; out = out + tn @ self.w[o]; tp, tc = tc, tn
        return (out + self.b).reshape(B, T, N, -1)

class STBlock(nn.Module):
    def __init__(self, cin, cout, k, drop):
        super().__init__()
        self.t1 = nn.Conv2d(cin, cout, (1, 3), padding=(0, 1)); self.t2 = nn.Conv2d(cout, cout, (1, 3), padding=(0, 1))
        self.g = ChebGraphConv(cout, cout, k, 0.5); self.n1 = nn.LayerNorm(cout); self.n2 = nn.LayerNorm(cout)
        self.d = nn.Dropout(drop); self.proj = nn.Linear(cin, cout) if cin != cout else None
    def forward(self, x, L):
        r = x
        h = F.gelu(self.t1(x.permute(0, 3, 2, 1))).permute(0, 3, 2, 1); h = self.d(self.n1(h))
        h = self.d(F.gelu(self.g(h, L)))
        h = self.t2(h.permute(0, 3, 2, 1)).permute(0, 3, 2, 1)
        if self.proj is not None: r = self.proj(r)
        return self.n2(h + r)

class AttnPool(nn.Module):
    def __init__(self, h): super().__init__(); self.s = nn.Sequential(nn.Linear(h, h//2), nn.Tanh(), nn.Linear(h//2, 1))
    def forward(self, x): w = F.softmax(self.s(x), 1); return (x*w).sum(1)

class LastPool(nn.Module):
    def forward(self, x): return x[:, -1]

class GraphNet(nn.Module):
    """ST-GCN (time feats enter with the flows) or MBC-GCN (separate time-embedding branch)."""
    def __init__(self, n_flow, n_time, hidden, layers, k, drop, attention=True, time_branch=False, time_dim=16):
        super().__init__(); self.n_flow = n_flow; self.tb = time_branch
        if time_branch:
            self.te = nn.Sequential(nn.Linear(n_time, time_dim), nn.GELU(), nn.Linear(time_dim, time_dim)); cin = n_flow + time_dim
        else: cin = n_flow + n_time
        self.inp = nn.Linear(cin, hidden)
        self.blocks = nn.ModuleList([STBlock(hidden, hidden, k, drop) for _ in range(layers)])
        self.pool = AttnPool(hidden) if attention else LastPool()
        self.out = nn.Sequential(nn.Linear(hidden, hidden//2), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden//2, 2))
    def forward(self, x, L):
        if self.tb: x = torch.cat([x[..., :self.n_flow], self.te(x[..., self.n_flow:])], -1)
        h = self.inp(x)
        for b in self.blocks: h = b(h, L)
        return self.out(self.pool(h))                      # (B, N, 2)

class NetLSTM(nn.Module):
    def __init__(self, n_nodes, n_feat, hidden, layers, drop, attention=False):
        super().__init__(); self.N = n_nodes
        self.lstm = nn.LSTM(n_nodes*n_feat, hidden, layers, batch_first=True, dropout=drop if layers > 1 else 0)
        self.pool = AttnPool(hidden) if attention else LastPool(); self.fc = nn.Linear(hidden, n_nodes*2)
    def forward(self, x, L=None):
        B, T, N, C = x.shape; h, _ = self.lstm(x.reshape(B, T, N*C)); return self.fc(self.pool(h)).reshape(B, N, 2)

def build_model(name, n_nodes, n_flow, n_time, hp, attention):
    if name == "LSTM":    return NetLSTM(n_nodes, n_flow+n_time, hp["hidden"], hp["layers"], hp["dropout"], attention)
    if name == "ST-GCN":  return GraphNet(n_flow, n_time, hp["hidden"], hp["layers"], hp["k"], hp["dropout"], attention, time_branch=False)
    if name == "MBC-GCN": return GraphNet(n_flow, n_time, hp["hidden"], hp["layers"], hp["k"], hp["dropout"], attention, time_branch=True, time_dim=hp.get("time_dim", 16))
    raise ValueError(name)

# --------------------------------------------------------------------------
# training / evaluation
# --------------------------------------------------------------------------
def train(model, tr, va, L, device, lr, wd, max_epochs, patience, node_mask=None, warmup=5, min_lr=1e-5):
    """AdamW with linear warm-up and cosine decay; early stopping on validation MSE."""
    model.to(device); L = L.to(device); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    floor_ = min_lr / lr
    def lr_lambda(ep):
        if ep < warmup: return (ep + 1) / warmup
        p = (ep - warmup) / max(1, max_epochs - warmup); return max(floor_, 0.5 * (1 + math.cos(math.pi * min(p, 1.0))))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    best, best_ep, best_state, bad = math.inf, 0, None, 0
    def loss_fn(pred, y):
        if node_mask is None: return F.mse_loss(pred, y)
        return F.mse_loss(pred[:, node_mask], y[:, node_mask])
    for ep in range(1, max_epochs + 1):
        model.train()
        for x, y in tr:
            x, y = x.to(device), y.to(device); opt.zero_grad(); loss = loss_fn(model(x, L), y); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        sched.step()
        model.eval(); vl, n = 0.0, 0
        with torch.no_grad():
            for x, y in va: x, y = x.to(device), y.to(device); vl += loss_fn(model(x, L), y).item()*len(x); n += len(x)
        vl /= max(n, 1)
        if not math.isfinite(vl): raise RuntimeError(f"non-finite validation loss on {device} at epoch {ep}")
        if vl < best - 1e-5: best, best_ep, bad, best_state = vl, ep, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    if best_state: model.load_state_dict(best_state)
    return best, ep, best_ep, bool(ep == max_epochs and best_ep > max_epochs - patience)

def train_safe(build, tr, va, L, device, lr, wd, max_epochs, patience, node_mask=None):
    """Train on `device`; if the loss goes non-finite (MPS numerical bug), rebuild and retrain on CPU.
    Returns (model, best_val, epochs, best_epoch, hit_cap, device_used)."""
    try:
        m = build(); best, ep, bep, cap = train(m, tr, va, L, device, lr, wd, max_epochs, patience, node_mask, WARMUP_EPOCHS, MIN_LR); return m, best, ep, bep, cap, str(device)
    except RuntimeError as e:
        if device.type == "cpu": raise
        print(f"   [fallback] {e} -> retraining on cpu")
        m = build(); best, ep, bep, cap = train(m, tr, va, L, torch.device("cpu"), lr, wd, max_epochs, patience, node_mask, WARMUP_EPOCHS, MIN_LR); return m, best, ep, bep, cap, "cpu"

def predict(model, loader, L, device):
    device = torch.device(device); model.to(device).eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in loader: ps.append(model(x.to(device), L.to(device)).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def metrics(y_true, y_pred, node_mask=None):
    """MAE / RMSE / R2 on counts, pooled over stations (or over the masked stations) and time, per target."""
    out = {}
    for j, nm in enumerate(["ride_in", "ride_out"]):
        t = y_true[..., j]; p = y_pred[..., j]
        if node_mask is not None: t, p = t[:, node_mask], p[:, node_mask]
        t, p = t.ravel(), p.ravel()
        ss = ((t - t.mean())**2).sum()
        out[nm] = {"MAE": float(np.abs(t-p).mean()), "RMSE": float(np.sqrt(((t-p)**2).mean())),
                   "R2": float(1 - ((t-p)**2).sum()/ss) if ss > 0 else np.nan}
    return out

def naive_baselines(X, w, splits, seq_len):
    """Counts-scale predictions for the test targets: persistence, seasonal-naive (same window previous day),
    and historical mean by (station, hour-of-day, weekday) from the train windows."""
    ste = set(int(i) for i in splits["test"])
    te = np.array([int(t) for t in splits["test"] if t - seq_len + 1 >= 0 and t + 1 < len(X) and (t + 1) in ste])
    y = X[te + 1, :, :2]
    hod = np.asarray(w.hour*60 + w.minute); wd = np.asarray(w.dayofweek < 5)
    tr = np.asarray(splits["train"]); keys = pd.Series([(int(a), bool(b)) for a, b in zip(hod[tr], wd[tr])])
    hist = {kk: X[tr[np.asarray(ix)], :, :2].mean(0) for kk, ix in keys.groupby(keys).indices.items()}
    fallback = X[tr, :, :2].mean(0)
    hm = np.stack([hist.get((int(hod[t+1]), bool(wd[t+1])), fallback) for t in te])
    steps_per_day = int((np.asarray(w.date) == w.date[0]).sum())
    sn = np.stack([X[t+1-steps_per_day, :, :2] if t+1-steps_per_day >= 0 else X[t, :, :2] for t in te])
    return y, {"persistence": X[te, :, :2], "seasonal_naive": sn, "hist_mean": hm}

## 7. Station-level forecasts with deep models (Tables 5, 7 and B.2)

Rolling-origin folds (train / 2 validation days / 2 test days); hyperparameters tuned on validation days only.

In [ ]:
# Configuration
import itertools, json, math, time, warnings, pickle
from pathlib import Path
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")

OUT_DIR   = FORECAST_DIR / "station"; OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUT_DIR / "results_all.csv"
TUNE_CSV    = OUT_DIR / "tune_runs.csv"

BUFFER    = 200
RES_LIST  = ["1h"]
FEATURES  = ["ar", "semi_ar"]
ATTENTION = [False]
MODELS    = ["LSTM", "ST-GCN", "MBC-GCN"]
SEEDS     = [0, 1, 2]
N_FOLDS, VAL_DAYS, TEST_DAYS = 3, 2, 2
FORCE_CPU = False

MAX_EPOCHS, PATIENCE = 200, 20
WARMUP_EPOCHS, MIN_LR = 5, 1e-5

SEQ_LEN_DEFAULT = {"15min": 12, "1h": 8}
HP = {
    "LSTM":    {"hidden": 160, "layers": 2, "dropout": 0.3,  "lr": 5e-4, "wd": 1e-6, "batch": 64},
    "ST-GCN":  {"hidden": 96,  "layers": 2, "k": 3, "dropout": 0.15, "lr": 5e-4, "wd": 1e-6, "batch": 32},
    "MBC-GCN": {"hidden": 96,  "layers": 2, "k": 3, "dropout": 0.15, "time_dim": 16, "lr": 5e-4, "wd": 1e-6, "batch": 32},
}

TUNE = True
N_TRIALS = {"LSTM": 15, "ST-GCN": 20, "MBC-GCN": 20}
TUNE_RES = ["1h"]
TUNE_FOLDS, TUNE_SEEDS = [0, 2], [0, 1]
TUNE_FEATURES, TUNE_ATTENTION = "semi_ar", False
SPACE_COMMON = {"hidden": ("cat", [64, 96, 128, 160]), "layers": ("cat", [1, 2, 3]), "dropout": ("float", 0.0, 0.4),
                "lr": ("log", 1e-4, 3e-3), "wd": ("log", 1e-7, 1e-3), "batch": ("cat", [16, 32, 64])}
SPACE_SEQ = {"1h": ("cat", [4, 8, 12]), "15min": ("cat", [8, 12, 24])}
SPACE_GRAPH = {"k": ("cat", [2, 3, 4])}

In [ ]:
# Data: tensors, graph, folds
device = torch.device("cpu") if FORCE_CPU else get_device(); print("device:", device)
data = {}
for res in RES_LIST:
    p = load_panel(PANEL_DIR / f"panel_{res}_{BUFFER}m.csv")
    X, w, nodes = panel_to_tensor(p)
    data[res] = dict(X=X, w=w, nodes=nodes, L=build_graph(nodes), folds=day_folds(w, VAL_DAYS, TEST_DAYS, N_FOLDS))
    print(f"[{res}] tensor {X.shape} | finite {np.isfinite(X).all()} | negative counts {(X[:,:,:4]<0).sum()} | zero share ride_out {(X[:,:,1]==0).mean():.1%}")
    for k, f in enumerate(data[res]["folds"]):
        print(f"   fold {k+1}: train {f['train'][0]}–{f['train'][-1]} | val {f['val'][0]}–{f['val'][-1]} | test {f['test'][0]}–{f['test'][-1]}")

In [ ]:
# Functions: one training run, naive references, cache
def feat_cols(features):
    flows = [0, 1] + ([2, 3] if features == "semi_ar" else []); return flows, list(range(4, 4 + len(TIME_FEATS)))

KEY = ["res", "fold", "features", "attention", "model", "seed"]
def hp_key(hp): return json.dumps({k: (round(v, 6) if isinstance(v, float) else v) for k, v in hp.items()}, sort_keys=True)

def run_one(res, fold_id, features, attention, model_name, seed, hp, final=True):
    """Train one configuration; final=False returns validation metrics only."""
    D = data[res]; X, w, L = D["X"], D["w"], D["L"]
    sp = split_index(w, D["folds"][fold_id]); sc = Scaler().fit(X, sp["train"]); Xs = sc.transform(X)
    flows, tcols = feat_cols(features); seq = hp.get("seq_len", SEQ_LEN_DEFAULT[res])
    tr, va, te = make_loaders(Xs, sp, seq, flows + tcols, hp["batch"])
    def build():
        set_seed(seed); return build_model(model_name, len(D["nodes"]), len(flows), len(tcols), hp, attention)
    t0 = time.time()
    m, best_val, epochs, best_ep, hit_cap, dev_used = train_safe(build, tr, va, L, device, hp["lr"], hp["wd"], MAX_EPOCHS, PATIENCE)
    row = {"res": res, "buffer_m": BUFFER, "fold": fold_id + 1, "features": features, "attention": attention, "model": model_name, "seed": seed,
           "hp": hp_key(hp), "val_mse": best_val, "epochs": epochs, "best_ep": best_ep, "hit_cap": hit_cap, "device": dev_used}
    if final:
        y, pr = predict(m, te, L, dev_used); mt = metrics(sc.inverse_targets(y), sc.inverse_targets(pr))
        for tgt, v in mt.items():
            for k, x in v.items(): row[f"{tgt}_{k}"] = x
    row["secs"] = round(time.time() - t0)
    return row

def run_naive(res, fold_id):
    D = data[res]; sp = split_index(D["w"], D["folds"][fold_id])
    y, nb = naive_baselines(D["X"], D["w"], sp, SEQ_LEN_DEFAULT[res]); rows = []
    for name, pred in nb.items():
        row = {"res": res, "buffer_m": BUFFER, "fold": fold_id + 1, "features": "-", "attention": False, "model": name, "seed": 0, "hp": "-", "device": "-"}
        for tgt, v in metrics(y, pred).items():
            for k, x in v.items(): row[f"{tgt}_{k}"] = x
        rows.append(row)
    return rows

def append_csv(path, df): df.to_csv(path, mode="a", header=not path.exists(), index=False)

def load_cache():
    """Reuse cached runs whose hyperparameters match HP_FINAL."""
    if not RESULTS_CSV.exists(): return {}
    old = pd.read_csv(RESULTS_CSV); old = old[old["model"].isin(MODELS) & np.isfinite(old["ride_out_MAE"])]
    old["attention"] = old["attention"].astype(str).str.lower().eq("true")
    return {tuple(r[k] for k in KEY): r.to_dict() for _, r in old.iterrows() if r["hp"] == hp_key(HP_FINAL[r["res"]][r["model"]])}

In [ ]:
# Hyperparameter tuning (validation days only)
try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING); HAVE_OPTUNA = True
except ImportError:
    HAVE_OPTUNA = False; print("optuna not installed (pip install optuna); falling back to random search")
tune_cache = {}
if TUNE_CSV.exists():
    for _, r in pd.read_csv(TUNE_CSV).iterrows(): tune_cache[(r["res"], r["model"], r["hp"], int(r["fold"]), int(r["seed"]))] = r.to_dict()

def draw(spec, trial=None, rng=None, name=""):
    kind = spec[0]
    if trial is not None:
        return trial.suggest_categorical(name, spec[1]) if kind == "cat" else trial.suggest_float(name, spec[1], spec[2], log=(kind == "log"))
    if kind == "cat": return spec[1][int(rng.integers(len(spec[1])))]
    return float(np.exp(rng.uniform(np.log(spec[1]), np.log(spec[2])))) if kind == "log" else float(rng.uniform(spec[1], spec[2]))

def space_for(res, model_name):
    return {**SPACE_COMMON, "seq_len": SPACE_SEQ[res], **(SPACE_GRAPH if model_name != "LSTM" else {})}

def tune_eval(res, model_name, hp):
    out = []
    for f in TUNE_FOLDS:
        for s in TUNE_SEEDS:
            key = (res, model_name, hp_key(hp), f + 1, s)
            if key not in tune_cache:
                row = run_one(res, f, TUNE_FEATURES, TUNE_ATTENTION, model_name, s, hp, final=False); tune_cache[key] = row; append_csv(TUNE_CSV, pd.DataFrame([row]))
            out.append(tune_cache[key])
    d = pd.DataFrame(out); return float(d["val_mse"].mean()), d

def diff_label(hp, base): return ", ".join(f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in hp.items() if base.get(k) != v) or "default"

def tune_model(res, model_name):
    base = {**HP[model_name], "seq_len": SEQ_LEN_DEFAULT[res]}; space = space_for(res, model_name); n_trials = N_TRIALS[model_name]
    print(f"[tune] {res} {model_name}: {n_trials} trials x {len(TUNE_FOLDS)} folds x {len(TUNE_SEEDS)} seeds"); T0 = time.time(); rows = []
    def evaluate(hp, i, label):
        val, d = tune_eval(res, model_name, hp)
        rows.append({"trial": i, "val": val, "seed_spread": d.groupby("fold")["val_mse"].agg(lambda x: x.max() - x.min()).mean(), "epochs": d["epochs"].mean(),
                     "hit_cap": int(d["hit_cap"].astype(bool).sum()), "label": label, **hp})
        print(f"  [{i:>2}] val {val:.4f} | ep {d['epochs'].mean():.0f} | {label}" + ("  [hit epoch cap]" if d["hit_cap"].astype(bool).any() else "")); return val
    def clean(hp):
        for k in ("hidden", "layers", "batch", "seq_len", "k"):
            if k in hp: hp[k] = int(hp[k])
        return hp
    if HAVE_OPTUNA:
        pkl = OUT_DIR / f"optuna_{res}_{model_name}.pkl"
        study = pickle.load(open(pkl, "rb")) if pkl.exists() else optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0, n_startup_trials=max(5, n_trials // 4)))
        if len(study.trials) == 0:
            study.enqueue_trial({k: base[k] for k, spec in space.items() if ((base[k] in spec[1]) if spec[0] == "cat" else (spec[1] <= base[k] <= spec[2]))})
        def objective(trial):
            hp = clean({**base, **{k: draw(spec, trial, name=k) for k, spec in space.items()}}); return evaluate(hp, trial.number, diff_label(hp, base))
        n_done = len([t for t in study.trials if t.state.is_finished()])
        study.optimize(objective, n_trials=max(0, n_trials - n_done), callbacks=[lambda st, tr: pickle.dump(st, open(pkl, "wb"))])
        if not rows:
            for t in study.trials: hp = clean({**base, **t.params}); evaluate(hp, t.number, diff_label(hp, base))
        best = clean({**base, **study.best_trial.params})
        try: print("  parameter importance:", {k: round(float(v), 3) for k, v in optuna.importance.get_param_importances(study).items()})
        except Exception: pass
    else:
        rng = np.random.default_rng(0); cands = [dict(base)] + [clean({**base, **{k: draw(spec, rng=rng) for k, spec in space.items()}}) for _ in range(n_trials - 1)]
        for i, hp in enumerate(cands): evaluate(hp, i, diff_label(hp, base))
        best = cands[int(np.argmin([r["val"] for r in rows]))]
    tab = pd.DataFrame(rows).sort_values("val").reset_index(drop=True); top = tab["val"].iloc[:5]; spread = tab["seed_spread"].iloc[0]
    print(f"  best {top.iloc[0]:.4f} vs default {tab[tab.trial == 0]['val'].iloc[0]:.4f} | gap best→5th {top.iloc[-1] - top.iloc[0]:.4f} vs seed spread {spread:.4f}"
          + ("  -> top trials differ by less than the seed spread: search is saturated" if top.iloc[-1] - top.iloc[0] < spread else "  -> trials still differ: N_TRIALS could be increased"))
    for k, spec in space.items():
        if spec[0] != "cat":
            lo, hi, v = spec[1], spec[2], best[k]; pos = (np.log(v / lo) / np.log(hi / lo)) if spec[0] == "log" else (v - lo) / (hi - lo)
            if pos < 0.05 or pos > 0.95: print(f"  ! {k}={v:.3g} is close to the search bound [{lo}, {hi}]; consider widening it")
    print(f"  took {(time.time()-T0)/60:.1f} min"); tab.to_csv(OUT_DIR / f"tune_table_{res}_{model_name}.csv", index=False)
    return best

HP_FINAL = {res: {m: {**HP[m], "seq_len": SEQ_LEN_DEFAULT[res]} for m in MODELS} for res in RES_LIST}
if TUNE:
    for res in TUNE_RES:
        for m in MODELS: HP_FINAL[res][m] = tune_model(res, m)
    for res in RES_LIST:
        if res not in TUNE_RES:
            src = TUNE_RES[0]; HP_FINAL[res] = {m: {**HP_FINAL[src][m], "seq_len": SEQ_LEN_DEFAULT[res]} for m in MODELS}
json.dump(HP_FINAL, open(OUT_DIR / "hp_final.json", "w"), indent=2)
print("\nfinal hyper-parameters:")
for res in RES_LIST: print(f"[{res}]"); print(pd.DataFrame.from_dict(HP_FINAL[res], orient="index").round(5).to_string())

In [ ]:
# Main loop (resumable)
cache = load_cache(); print(f"{len(cache)} valid runs already cached (matching the current hyper-parameters)")
todo = [(res, f, feat, att, mdl, sd) for res in RES_LIST for f in range(N_FOLDS)
        for feat, att, mdl, sd in itertools.product(FEATURES, ATTENTION, MODELS, SEEDS) if (res, f + 1, feat, att, mdl, sd) not in cache]
print(f"{len(todo)} runs to do"); T0 = time.time()
for i, (res, f, feat, att, mdl, sd) in enumerate(todo, 1):
    row = run_one(res, f, feat, att, mdl, sd, HP_FINAL[res][mdl]); append_csv(RESULTS_CSV, pd.DataFrame([row]))
    el = time.time() - T0
    print(f"[{i:>3}/{len(todo)}] {res} f{f+1} {feat:8s} attn={att!s:5s} {mdl:8s} s{sd} | MAE in/out {row['ride_in_MAE']:.3f}/{row['ride_out_MAE']:.3f} "
          f"| ep {row['epochs']} (best {row['best_ep']})" + ("  [hit epoch cap]" if row["hit_cap"] else "") + f" | {row['device']} {row['secs']}s | eta {el/i*(len(todo)-i)/60:.0f} min")
cache = load_cache(); naive = [r for res in RES_LIST for f in range(N_FOLDS) for r in run_naive(res, f)]
res_df = pd.DataFrame(list(cache.values()) + naive)
print("runs:", len(cache), "| non-finite:", int((~np.isfinite(res_df[res_df["model"].isin(MODELS)]["ride_out_MAE"])).sum()))
conv = res_df[res_df["model"].isin(MODELS)].groupby(["res", "model"]).agg(runs=("epochs", "size"), epochs_mean=("epochs", "mean"), best_ep_mean=("best_ep", "mean"),
                                                                           hit_cap=("hit_cap", lambda x: int(x.astype(bool).sum()))).round(1)
print(conv.to_string()); n_cap = int(conv["hit_cap"].sum())
print(f"{n_cap} runs hit MAX_EPOCHS={MAX_EPOCHS} while still improving" + ("  -> increase MAX_EPOCHS and rerun" if n_cap else "  -> all runs converged"))

In [ ]:
# Summary table
def summarise(df):
    g = df.groupby(["res", "model", "features", "attention"])
    m = g[["ride_in_MAE", "ride_out_MAE", "ride_in_RMSE", "ride_out_RMSE", "ride_in_R2", "ride_out_R2"]].agg(["mean", "std"])
    m.columns = [f"{a}_{b}" for a, b in m.columns]; m["n_runs"] = g.size(); return m.round(3).reset_index()

summary = summarise(res_df); summary.to_csv(OUT_DIR / "summary.csv", index=False)
for res in RES_LIST:
    print(f"\n===== {res} × {BUFFER} m  (mean ± sd over {N_FOLDS} folds × {len(SEEDS)} seeds) =====")
    s = summary[summary["res"] == res].sort_values(["model", "features", "attention"])
    print(s[["model", "features", "attention", "ride_in_MAE_mean", "ride_in_MAE_std", "ride_out_MAE_mean", "ride_out_MAE_std",
             "ride_in_R2_mean", "ride_out_R2_mean", "n_runs"]].to_string(index=False))

In [ ]:
# Metro contribution: AR minus Semi-AR, paired within (fold, seed, model); one-sided Wilcoxon
from scipy.stats import wilcoxon
key = ["res", "fold", "seed", "model", "attention"]
ar = res_df[res_df["features"] == "ar"].set_index(key); sa = res_df[res_df["features"] == "semi_ar"].set_index(key)
d = (ar[["ride_in_MAE", "ride_out_MAE"]] - sa[["ride_in_MAE", "ride_out_MAE"]]).dropna()
d["ride_in_%"] = 100 * d["ride_in_MAE"] / ar.loc[d.index, "ride_in_MAE"]; d["ride_out_%"] = 100 * d["ride_out_MAE"] / ar.loc[d.index, "ride_out_MAE"]
gain = d.groupby(["res", "model", "attention"]).agg(["mean", "std", "count"]).round(3)
print("metro information gain (positive = Semi-AR better):"); print(gain.to_string()); gain.to_csv(OUT_DIR / "metro_gain.csv")
for (res, model, attn), g in d.groupby(["res", "model", "attention"]):
    if len(g) >= 5:
        print(f"{res} {model} attn={attn}: Wilcoxon p(ride_out gain>0) = {wilcoxon(g['ride_out_MAE'], alternative='greater').pvalue:.3f}")

## 8. Statistical core, SC-LSTM hybrid and forecast combination (Tables 6 and 9, Figure 10)

8a retrains the station-level LSTM (Semi-AR) and saves its test predictions; 8b-8d build the statistical core, train the hybrid (core as offset, LSTM on the residual) and compare.

In [ ]:
# Configuration (hyperparameters are read from Section 7)
import itertools, json, math, time, warnings, pickle
from pathlib import Path
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")

OUT_DIR   = FORECAST_DIR / "station"; OUT_DIR.mkdir(parents=True, exist_ok=True)
BUFFER    = 200
RES_LIST  = ["1h"]
MODELS    = ["LSTM", "ST-GCN", "MBC-GCN"]
N_FOLDS, VAL_DAYS, TEST_DAYS = 3, 2, 2
FORCE_CPU = False
MAX_EPOCHS, PATIENCE = 200, 20
WARMUP_EPOCHS, MIN_LR = 5, 1e-5
SEQ_LEN_DEFAULT = {"15min": 12, "1h": 8}
HP = {
    "LSTM":    {"hidden": 160, "layers": 2, "dropout": 0.3,  "lr": 5e-4, "wd": 1e-6, "batch": 64},
    "ST-GCN":  {"hidden": 96,  "layers": 2, "k": 3, "dropout": 0.15, "lr": 5e-4, "wd": 1e-6, "batch": 32},
    "MBC-GCN": {"hidden": 96,  "layers": 2, "k": 3, "dropout": 0.15, "time_dim": 16, "lr": 5e-4, "wd": 1e-6, "batch": 32},
}
hp_path = OUT_DIR / "hp_final.json"
if hp_path.exists(): HP_FINAL = json.load(open(hp_path)); print("HP_FINAL read from", hp_path)
else:
    HP_FINAL = {res: {m: {**HP[m], "seq_len": SEQ_LEN_DEFAULT[res]} for m in MODELS} for res in RES_LIST}
    print(f"! {hp_path} not found; using default hyperparameters (run the station-level section first)")

In [ ]:
# Data: tensors, graph, folds
device = torch.device("cpu") if FORCE_CPU else get_device(); print("device:", device)
data = {}
for res in RES_LIST:
    p = load_panel(PANEL_DIR / f"panel_{res}_{BUFFER}m.csv")
    X, w, nodes = panel_to_tensor(p)
    data[res] = dict(X=X, w=w, nodes=nodes, L=build_graph(nodes), folds=day_folds(w, VAL_DAYS, TEST_DAYS, N_FOLDS))
    print(f"[{res}] tensor {X.shape} | finite {np.isfinite(X).all()} | negative counts {(X[:,:,:4]<0).sum()} | zero share ride_out {(X[:,:,1]==0).mean():.1%}")
    for k, f in enumerate(data[res]["folds"]):
        print(f"   fold {k+1}: train {f['train'][0]}–{f['train'][-1]} | val {f['val'][0]}–{f['val'][-1]} | test {f['test'][0]}–{f['test'][-1]}")

### 8a. LSTM test predictions

In [ ]:
# LSTM (Semi-AR): train and save test predictions
RES7, FEAT7, ATTN7, SEEDS7 = "1h", "semi_ar", False, [0, 1, 2]
OUT7 = OUT_DIR / "lstm_full"; OUT7.mkdir(parents=True, exist_ok=True)
RES7_CSV = OUT7 / "results_lstm_full.csv"
HP7 = HP_FINAL[RES7]["LSTM"]
D7 = data[RES7]; X7, w7 = D7["X"], D7["w"]

def feat_cols7(features):
    flows = [0, 1] + ([2, 3] if features == "semi_ar" else []); return flows, list(range(4, 4 + len(TIME_FEATS)))

def pred7_path(f, sd): return OUT7 / f"pred_LSTM_full_f{f+1}_s{sd}.npz"

def run_lstm_full(fold_id, seed):
    """Train one (fold, seed); returns the metrics row and saves predictions to npz."""
    sp = split_index(w7, D7["folds"][fold_id]); sc = Scaler().fit(X7, sp["train"]); Xs = sc.transform(X7)
    flows, tcols = feat_cols7(FEAT7); hp = HP7; seq = hp.get("seq_len", SEQ_LEN_DEFAULT[RES7]); L = D7["L"]
    tr, va, te = make_loaders(Xs, sp, seq, flows + tcols, hp["batch"])
    def build(): set_seed(seed); return build_model("LSTM", X7.shape[1], len(flows), len(tcols), hp, ATTN7)
    t0 = time.time(); m, best_val, epochs, best_ep, hit_cap, dev_used = train_safe(build, tr, va, L, device, hp["lr"], hp["wd"], MAX_EPOCHS, PATIENCE)
    y, p = predict(m, te, L, dev_used); y, p = sc.inverse_targets(y), sc.inverse_targets(p)
    row = {"fold": fold_id + 1, "seed": seed, "val_mse": best_val, "epochs": epochs, "best_ep": best_ep, "hit_cap": hit_cap, "secs": round(time.time() - t0)}
    for tgt, v in metrics(y, p).items():
        for k, x in v.items(): row[f"{tgt}_{k}"] = x
    te_t = np.array([t + 1 for t in sp["test"] if t - seq + 1 >= 0 and t + 1 < len(X7) and (t + 1) in set(int(i) for i in sp["test"])])
    np.savez_compressed(pred7_path(fold_id, seed), y=y, p=p, te_t=te_t, station_mae=np.abs(y - p).mean(0))
    return row

done = {}
if RES7_CSV.exists():
    old = pd.read_csv(RES7_CSV)
    done = {(int(r["fold"]), int(r["seed"])): r.to_dict() for _, r in old.iterrows() if np.isfinite(r["ride_out_MAE"]) and pred7_path(int(r["fold"]) - 1, int(r["seed"])).exists()}
todo = [(f, sd) for f in range(N_FOLDS) for sd in SEEDS7 if (f + 1, sd) not in done]
print(f"{len(done)} runs cached | {len(todo)} to do")
for i, (f, sd) in enumerate(todo, 1):
    row = run_lstm_full(f, sd); done[(f + 1, sd)] = row
    pd.DataFrame([row]).to_csv(RES7_CSV, mode="a", header=not RES7_CSV.exists(), index=False)
    print(f"[{i}/{len(todo)}] fold {f+1} seed {sd} | MAE in/out {row['ride_in_MAE']:.3f}/{row['ride_out_MAE']:.3f} | epochs {row['epochs']} (best {row['best_ep']})")

### 8b. Statistical core

In [ ]:
# Configuration
import itertools, json, math, time, warnings, pickle
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
from scipy.stats import nbinom, wilcoxon
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

RES9   = "1h"
OUT9   = OUT_DIR / "hybrid"; OUT9.mkdir(parents=True, exist_ok=True)
SEEDS9 = [0, 1, 2]
SPAT_PRED_DIR = OUT_DIR / "lstm_full"
EMPTY_THRESH, GROUP_EDGES, FLOOR, SHORT_H = 0.05, (0.5, 2.0), 0.01, 3

GRID = {"half_life": [1, 2, 3, 5, np.inf], "dshrink": [0, 2, 8], "k": [1, 3, 10, 30, 100], "ratio_prior": [5, 20, 40, 80], "beta": [0.0, 0.25, 0.5, 0.75, 1.0]}

HP_HYB = {"hidden": 64, "layers": 1, "dropout": 0.1, "lr": 1e-3, "wd": 1e-5, "batch": 16, "seq_len": 8, "f_pen": 1.0, "alpha": 0.5}
MAX_EPOCHS9, PATIENCE9, WARMUP9, MIN_LR9 = 200, 20, 5, 1e-5
TUNE9, N_TRIALS9, TUNE_FOLDS9, TUNE_SEEDS9 = True, 15, [0, 2], [0]
SPACE_HYB = {"hidden": ("cat", [32, 64, 128]), "dropout": ("float", 0.0, 0.4), "lr": ("log", 3e-4, 5e-3), "wd": ("log", 1e-7, 1e-3),
             "f_pen": ("log", 0.1, 30.0), "alpha": ("cat", [0.0, 0.5, 1.0]), "seq_len": ("cat", [4, 8, 12])}
try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING); HAVE_OPTUNA = True
except ImportError: HAVE_OPTUNA = False; print("optuna not installed; falling back to random search")

D9 = data[RES9]; X9, w9 = D9["X"], D9["w"]; T9, N9 = X9.shape[:2]; Y9 = np.asarray(X9[:, :, :2], float)
dates9 = pd.to_datetime(pd.Series(np.asarray(w9.date))).dt.normalize().to_numpy(); udays9 = np.unique(dates9); day9 = np.searchsorted(udays9, dates9)
SPD9 = int((day9 == 0).sum()); first9 = np.searchsorted(day9, np.arange(len(udays9))); slot9 = np.arange(T9) - first9[day9]; iswk9 = (pd.DatetimeIndex(udays9).weekday >= 5).astype(int)
nd9 = len(udays9); Yd9 = Y9.reshape(nd9, SPD9, N9, 2)
print(f"{RES9}: {nd9} days x {SPD9} steps x {N9} stations | device {device}")

In [ ]:
# Functions: profile, empirical-Bayes shrinkage, level ratio, dispersion; statistical core per fold
def build_profiles(hl, ds):
    i = np.arange(nd9); lag = i[:, None] - i[None, :] - 1
    wgt = np.where(lag >= 0, (0.5 ** (np.maximum(lag, 0) / hl)) if np.isfinite(hl) else 1.0, 0.0); Wdt = wgt * (iswk9[:, None] == iswk9[None, :])
    raw_all = np.einsum("di,ihsc->dhsc", wgt, Yd9) / np.maximum(wgt.sum(1), 1e-12)[:, None, None, None]
    raw_dt = np.einsum("di,ihsc->dhsc", Wdt, Yd9) / np.maximum(Wdt.sum(1), 1e-12)[:, None, None, None]
    Wd = Wdt.sum(1); den = (Wd + ds)[:, None, None, None]
    return np.where(den > 0, (Wd[:, None, None, None] * raw_dt + ds * raw_all) / np.maximum(den, 1e-12), raw_all), np.minimum(Wd + ds, wgt.sum(1))

def eb_shrink(prof, W_eff, k):
    level = prof.mean(1, keepdims=True); net = prof.mean(2, keepdims=True); shape = net / np.maximum(net.mean(1, keepdims=True), 1e-9)
    m = np.maximum(level * shape, 1e-6); W = W_eff[:, None, None, None]; return (k + W * prof) / (k / m + W)

def _winsum(a, L):
    c = np.cumsum(np.concatenate([np.zeros((1,) + a.shape[1:]), a], 0), 0); t = np.arange(len(a)); out = np.zeros_like(a); ok = t - L >= 0
    out[ok] = c[t[ok]] - c[t[ok] - L]; return out

def level_ratio(B, prior, beta):
    rd = (_winsum(Y9, SPD9) + prior) / (_winsum(B, SPD9) + prior); ps = prior * SHORT_H / SPD9
    rs = (_winsum(Y9, SHORT_H) + ps) / (_winsum(B, SHORT_H) + ps); return rd ** (1 - beta) * rs ** beta

def pois_dev(y, mu):
    mu = np.maximum(mu, FLOOR); return float(np.mean(2 * (np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0.0) - (y - mu))))

def fit_phi(y, mu):
    y = y.ravel(); mu = np.maximum(mu.ravel(), FLOOR)
    def nll(lp):
        phi = math.exp(lp); return -np.mean(gammaln(y + phi) - gammaln(phi) - gammaln(y + 1) + phi * np.log(phi / (phi + mu)) + y * np.log(mu / (phi + mu)))
    return math.exp(minimize_scalar(nll, bounds=(-3, 7), method="bounded").x)

def targets_of(idx, S=1):
    s = set(int(i) for i in idx); return np.array(sorted(t + 1 for t in idx if (t + 1) in s and t - S + 1 >= 0 and t + 1 < T9), dtype=int)

def stat_core(fold_id):
    """Statistical core of a fold: MU (T,N,2), B, phi (N,2), lvl, grp, params; parameters chosen on training and validation days."""
    sp = split_index(w9, D9["folds"][fold_id]); tv = np.r_[targets_of(sp["train"]), targets_of(sp["val"])]; tv = tv[day9[tv] >= 2]
    lvl = Y9[sp["train"]].mean(0); best = None
    for hl, ds in itertools.product(GRID["half_life"], GRID["dshrink"]):
        prof, W = build_profiles(hl, ds)
        for k in GRID["k"]:
            B = eb_shrink(prof, W, k).reshape(T9, N9, 2)
            for pr, be in itertools.product(GRID["ratio_prior"], GRID["beta"]):
                dev = pois_dev(Y9[tv], (B * level_ratio(B, pr, be))[tv])
                if best is None or dev < best[0]: best = (dev, dict(half_life=hl, dshrink=ds, k=k, ratio_prior=pr, beta=be))
    bp = best[1]; prof, W = build_profiles(bp["half_life"], bp["dshrink"]); B = eb_shrink(prof, W, bp["k"]).reshape(T9, N9, 2); MU = B * level_ratio(B, bp["ratio_prior"], bp["beta"])
    grp = np.digitize(lvl, GROUP_EDGES); phi = np.ones((N9, 2))
    for g in range(3):
        mk = grp == g
        if mk.any(): phi[mk] = fit_phi(Y9[tv][:, mk], MU[tv][:, mk])
    return dict(sp=sp, MU=MU, B=B, phi=phi, lvl=lvl, grp=grp, params=bp)

CORE_PKL = OUT9 / "stat_core.pkl"
if CORE_PKL.exists(): cores = pickle.load(open(CORE_PKL, "rb")); print("statistical core read from cache")
else:
    T0 = time.time(); cores = {}
    for f in range(N_FOLDS): cores[f] = stat_core(f); print(f"fold {f+1}: {cores[f]['params']} | φ {np.round([np.unique(cores[f]['phi'][cores[f]['grp']==g])[0] for g in range(3)], 1)} | {time.time()-T0:.0f}s")
    pickle.dump(cores, open(CORE_PKL, "wb"))

### 8c. SC-LSTM hybrid

In [ ]:
# Functions: hybrid data set, model, training, tuning
def nb_nll_t(y, log_mu, phi):
    mu = log_mu.exp(); return -(torch.lgamma(y + phi) - torch.lgamma(phi) - torch.lgamma(y + 1) + phi * (torch.log(phi) - torch.log(phi + mu)) + y * (log_mu - torch.log(phi + mu)))

class HybDataset(Dataset):
    """x: [scaled flows 4, deviation from profile 2, time features 5, log level ratio 2] -> (T,N,13); y: target counts (N,2); off: log MU[t+1] (N,2)."""
    def __init__(self, Xs, dev, rec, idx, seq, MU):
        self.Xs, self.dev, self.rec, self.seq, self.MU = Xs, dev, rec, seq, np.log(np.maximum(MU, FLOOR)).astype(np.float32)
        s = set(int(i) for i in idx); self.t = [int(t) for t in idx if t - seq + 1 >= 0 and t + 1 < T9 and (t + 1) in s]
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        t = self.t[i]; sl = slice(t - self.seq + 1, t + 1)
        x = np.concatenate([self.Xs[sl], self.dev[sl], np.repeat(self.rec[t + 1][None], self.seq, 0)], 2).astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(Y9[t + 1].astype(np.float32)), torch.from_numpy(self.MU[t + 1])

class SCLSTM(nn.Module):
    def __init__(self, n_nodes, n_feat, hidden, layers, drop):
        super().__init__(); self.N = n_nodes
        self.lstm = nn.LSTM(n_nodes * n_feat, hidden, layers, batch_first=True, dropout=drop if layers > 1 else 0); self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hidden, n_nodes * 2); nn.init.zeros_(self.fc.weight); nn.init.zeros_(self.fc.bias)
    def forward(self, x):
        B, T, N, C = x.shape; h, _ = self.lstm(x.reshape(B, T, N * C)); return self.fc(self.drop(h[:, -1])).reshape(B, N, 2)

def make_hyb_loaders(fold_id, hp):
    c = cores[fold_id]; sp = c["sp"]; sc = Scaler().fit(X9, sp["train"]); Xs = sc.transform(X9)
    dev = (np.log1p(Y9) - np.log1p(np.maximum(c["B"], 0))).astype(np.float32); sd = dev[sp["train"]].std() + 1e-6; dev = dev / sd
    rec = np.log(np.maximum(c["MU"], FLOOR) / np.maximum(c["B"], FLOOR)).astype(np.float32); rec = rec / (rec[sp["train"]].std() + 1e-6)
    S = hp["seq_len"]; ds = lambda idx, shuf: DataLoader(HybDataset(Xs, dev, rec, idx, S, c["MU"]), hp["batch"], shuffle=shuf)
    return ds(sp["train"], True), ds(sp["val"], False), ds(sp["test"], False), c

def eval_nll_loader(m, loader, phi_t, dev_, mask_t):
    m.eval(); tot, n = 0.0, 0
    with torch.no_grad():
        for x, y, off in loader:
            x, y, off = x.to(dev_), y.to(dev_), off.to(dev_); nll = nb_nll_t(y, off + m(x), phi_t)[:, mask_t].mean(); tot += nll.item() * len(x); n += len(x)
    return tot / max(n, 1)

def train_hyb(fold_id, hp, seed, final=True):
    tr, va, te, c = make_hyb_loaders(fold_id, hp); dev_ = device
    phi_t = torch.tensor(c["phi"], dtype=torch.float32, device=dev_); mask = torch.tensor(c["lvl"] >= EMPTY_THRESH, device=dev_)
    w_s = (c["lvl"] + 0.5) ** (-hp["alpha"]); w_s = torch.tensor(w_s / w_s.mean(), dtype=torch.float32, device=dev_)
    set_seed(seed); m = SCLSTM(N9, 13, hp["hidden"], hp["layers"], hp["dropout"]).to(dev_)
    opt = torch.optim.AdamW(m.parameters(), lr=hp["lr"], weight_decay=hp["wd"]); floor_ = MIN_LR9 / hp["lr"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda ep: (ep + 1) / WARMUP9 if ep < WARMUP9 else max(floor_, 0.5 * (1 + math.cos(math.pi * min(1.0, (ep - WARMUP9) / max(1, MAX_EPOCHS9 - WARMUP9))))))
    best, best_ep, best_state, bad, t0 = math.inf, 0, None, 0, time.time()
    base_val = eval_nll_loader(m, va, phi_t, dev_, mask)
    for ep in range(1, MAX_EPOCHS9 + 1):
        m.train()
        for x, y, off in tr:
            x, y, off = x.to(dev_), y.to(dev_), off.to(dev_); f = m(x); loss = nb_nll_t(y, off + f, phi_t)[:, mask].mean() + hp["f_pen"] * (w_s * (f ** 2)).mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 5.0); opt.step()
        sched.step(); vl = eval_nll_loader(m, va, phi_t, dev_, mask)
        if not math.isfinite(vl): raise RuntimeError("non-finite val NLL")
        if vl < best - 1e-5: best, best_ep, bad, best_state = vl, ep, 0, {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE9: break
    if best_state: m.load_state_dict(best_state)
    row = {"fold": fold_id + 1, "seed": seed, "hp": json.dumps(hp, sort_keys=True), "val_nll": best, "val_nll_core": base_val, "epochs": ep, "best_ep": best_ep, "secs": round(time.time() - t0)}
    if not final: return row, None
    m.eval(); ys, mus, fs = [], [], []
    with torch.no_grad():
        for x, y, off in te: f = m(x.to(dev_)).cpu(); fs.append(f.numpy()); mus.append(torch.exp(off + f).numpy()); ys.append(y.numpy())
    y_te, mu_te, f_te = np.concatenate(ys), np.concatenate(mus), np.concatenate(fs); row["f_rms"] = float(np.sqrt((f_te ** 2).mean()))
    return row, (y_te, mu_te)

def draw(spec, trial=None, rng=None, name=""):
    k = spec[0]
    if trial is not None: return trial.suggest_categorical(name, spec[1]) if k == "cat" else trial.suggest_float(name, spec[1], spec[2], log=(k == "log"))
    if k == "cat": return spec[1][int(rng.integers(len(spec[1])))]
    return float(np.exp(rng.uniform(np.log(spec[1]), np.log(spec[2])))) if k == "log" else float(rng.uniform(spec[1], spec[2]))
def clean(hp):
    for k in ("hidden", "layers", "batch", "seq_len"): hp[k] = int(hp[k])
    return hp
tune_csv = OUT9 / "tune_hyb.csv"; tcache = {}
if tune_csv.exists():
    for _, r in pd.read_csv(tune_csv).iterrows(): tcache[(r["hp"], int(r["fold"]), int(r["seed"]))] = float(r["val_nll"])
def tune_eval(hp):
    vals = []
    for f in TUNE_FOLDS9:
        for s in TUNE_SEEDS9:
            key = (json.dumps(hp, sort_keys=True), f + 1, s)
            if key not in tcache:
                row, _ = train_hyb(f, hp, s, final=False); tcache[key] = row["val_nll"]; pd.DataFrame([row]).to_csv(tune_csv, mode="a", header=not tune_csv.exists(), index=False)
            vals.append(tcache[key])
    return float(np.mean(vals))
HP_HYB_FINAL = dict(HP_HYB)
if TUNE9:
    print(f"[tune] SC-LSTM: {N_TRIALS9} trials x {len(TUNE_FOLDS9)} folds x {len(TUNE_SEEDS9)} seeds"); T0 = time.time(); rows = []
    base = dict(HP_HYB)
    def evaluate(hp, i):
        v = tune_eval(hp); rows.append({"trial": i, "val_nll": v, **hp}); print(f"  [{i:>2}] val NLL {v:.4f} | " + ", ".join(f"{k}={v_:.3g}" if isinstance(v_, float) else f"{k}={v_}" for k, v_ in hp.items() if base.get(k) != v_) or "default"); return v
    if HAVE_OPTUNA:
        pkl = OUT9 / "optuna_hyb.pkl"; study = pickle.load(open(pkl, "rb")) if pkl.exists() else optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0, n_startup_trials=5))
        if len(study.trials) == 0: study.enqueue_trial({k: base[k] for k, spec in SPACE_HYB.items() if ((base[k] in spec[1]) if spec[0] == "cat" else (spec[1] <= base[k] <= spec[2]))})
        study.optimize(lambda tr: evaluate(clean({**base, **{k: draw(spec, tr, name=k) for k, spec in SPACE_HYB.items()}}), tr.number), n_trials=max(0, N_TRIALS9 - len(study.trials)), callbacks=[lambda st, tr: pickle.dump(st, open(pkl, "wb"))])
        HP_HYB_FINAL = clean({**base, **study.best_trial.params})
    else:
        rng = np.random.default_rng(0); cands = [dict(base)] + [clean({**base, **{k: draw(spec, rng=rng) for k, spec in SPACE_HYB.items()}}) for _ in range(N_TRIALS9 - 1)]
        for i, hp in enumerate(cands): evaluate(hp, i)
        HP_HYB_FINAL = cands[int(np.argmin([r["val_nll"] for r in rows]))]
    print(f"  took {(time.time()-T0)/60:.1f} min"); pd.DataFrame(rows).to_csv(OUT9 / "tune_table_hyb.csv", index=False)
json.dump(HP_HYB_FINAL, open(OUT9 / "hp_hyb_final.json", "w"), indent=2); print("SC-LSTM final hyperparameters:", HP_HYB_FINAL)

In [ ]:
# Main loop and summary table
def eval_counts(y, mu, phi=None, mask=None):
    out = {}
    for j, tgt in enumerate(("ride_in", "ride_out")):
        yt, pt = y[:, :, j], np.maximum(mu[:, :, j], 0)
        if mask is not None: yt, pt = yt[:, mask[:, j]], pt[:, mask[:, j]]
        e = (yt - pt).ravel(); out[f"{tgt}_MAE"] = np.abs(e).mean(); out[f"{tgt}_RMSE"] = np.sqrt((e ** 2).mean()); out[f"{tgt}_WMAPE"] = np.abs(e).sum() / yt.sum()
        out[f"{tgt}_bias_ratio"] = pt.sum() / yt.sum(); out[f"{tgt}_R2"] = 1 - (e ** 2).sum() / ((yt - yt.mean()) ** 2).sum()
        if phi is not None:
            ph = phi[:, j][mask[:, j]] if mask is not None else phi[:, j]; ph = np.broadcast_to(ph, yt.shape); m_ = np.maximum(pt, FLOOR); pp = ph / (ph + m_)
            out[f"{tgt}_NLL"] = -nbinom.logpmf(yt, ph, pp).mean(); out[f"{tgt}_cov90"] = ((yt >= nbinom.ppf(0.05, ph, pp)) & (yt <= nbinom.ppf(0.95, ph, pp))).mean()
    return out

res_csv = OUT9 / "results_hyb.csv"; done = {}
if res_csv.exists():
    for _, r in pd.read_csv(res_csv).iterrows():
        if r["hp"] == json.dumps(HP_HYB_FINAL, sort_keys=True) and (OUT9 / f"pred_hyb_f{int(r['fold'])}_s{int(r['seed'])}.npz").exists(): done[(int(r["fold"]), int(r["seed"]))] = r.to_dict()
todo = [(f, s) for f in range(N_FOLDS) for s in SEEDS9 if (f + 1, s) not in done]; print(f"{len(done)} cached | {len(todo)} to do"); T0 = time.time()
for i, (f, s) in enumerate(todo, 1):
    row, (y_te, mu_te) = train_hyb(f, HP_HYB_FINAL, s); np.savez_compressed(OUT9 / f"pred_hyb_f{f+1}_s{s}.npz", y=y_te, mu=mu_te)
    row.update(eval_counts(y_te, mu_te, cores[f]["phi"], cores[f]["lvl"] >= EMPTY_THRESH)); pd.DataFrame([row]).to_csv(res_csv, mode="a", header=not res_csv.exists(), index=False); done[(f + 1, s)] = row
    print(f"[{i}/{len(todo)}] f{f+1} s{s} | val NLL {row['val_nll']:.4f} (core {row['val_nll_core']:.4f}) | test MAE out {row['ride_out_MAE']:.3f} NLL {row['ride_out_NLL']:.4f} | f_rms {row['f_rms']:.3f} | ep {row['epochs']} (best {row['best_ep']}) | {row['secs']}s | {(time.time()-T0)/60:.1f} min")

rows, pooled = [], {"y": [], "core": [], "hyb": [], "lstm": [], "lvl": [], "phi": []}
S_hyb = HP_HYB_FINAL["seq_len"]
for f in range(N_FOLDS):
    c = cores[f]; te_t = targets_of(c["sp"]["test"], S_hyb); y = Y9[te_t]; mask = c["lvl"] >= EMPTY_THRESH; preds = {"stat_core": c["MU"][te_t]}
    hyb = [np.load(OUT9 / f"pred_hyb_f{f+1}_s{s}.npz") for s in SEEDS9]; assert all(np.allclose(h["y"], y, atol=0.5) for h in hyb), "hybrid-model test targets are not aligned"
    preds["SC-LSTM"] = np.mean([h["mu"] for h in hyb], 0)
    lp = [p for p in (SPAT_PRED_DIR / f"pred_LSTM_full_f{f+1}_s{s}.npz" for s in range(10)) if p.exists()]
    if lp:
        z = [np.load(p) for p in lp]; p_l = np.mean([zz["p"] for zz in z], 0); y_l = z[0]["y"]
        if y_l.shape == y.shape and np.allclose(y_l, y, atol=0.5): preds["LSTM_full"] = p_l
        else:
            te7 = z[0]["te_t"] if "te_t" in z[0] else None
            if te7 is not None:
                pos = {t: i for i, t in enumerate(te7)}; sel = [pos[t] for t in te_t if t in pos]
                if len(sel) == len(te_t): preds["LSTM_full"] = p_l[sel]
    if "LSTM_full" in preds:
        preds["ens_equal"] = (preds["stat_core"] + preds["LSTM_full"]) / 2
        w = np.clip((np.log(c["lvl"] + 0.5) - np.log(0.5)) / (np.log(2.5) - np.log(0.5)), 0, 1)
        preds["ens_flow"] = (1 - w)[None] * preds["stat_core"] + w[None] * preds["LSTM_full"]
        preds["ens_hyb_lstm"] = (preds["SC-LSTM"] + preds["LSTM_full"]) / 2
    for name, p in preds.items():
        rows.append({"fold": f + 1, "model": name, **eval_counts(y, p, c["phi"] if name in ("stat_core", "SC-LSTM") else None, mask)})
    pooled["y"].append(y); pooled["core"].append(preds["stat_core"]); pooled["hyb"].append(preds["SC-LSTM"]); pooled["lstm"].append(preds.get("LSTM_full")); pooled["lvl"].append(c["lvl"]); pooled["phi"].append(c["phi"])
sum_df = pd.DataFrame(rows); cols = ["ride_in_MAE", "ride_out_MAE", "ride_out_RMSE", "ride_out_WMAPE", "ride_out_R2", "ride_out_bias_ratio", "ride_out_NLL", "ride_out_cov90"]
tab = sum_df.groupby("model")[cols].mean(); tab["ride_out_MAE_sd"] = sum_df.groupby("model")["ride_out_MAE"].std(ddof=0)
order = [m for m in ["stat_core", "LSTM_full", "SC-LSTM", "ens_equal", "ens_flow", "ens_hyb_lstm"] if m in tab.index]; tab = tab.loc[order]
print(f"\n===== non-empty stations (training mean >= {EMPTY_THRESH}), 3-fold average =====\n" + tab.round(3).to_string()); tab.to_csv(OUT9 / "summary_hybrid.csv")
hy = sum_df[sum_df.model == "SC-LSTM"]; rr = pd.read_csv(res_csv); rr = rr[rr["hp"] == json.dumps(HP_HYB_FINAL, sort_keys=True)]
print(f"\nRMS of the SC-LSTM residual correction f (seed average): {rr['f_rms'].mean():.3f}  (about 0 = network keeps the statistical core; 0.1 = corrections of about 10%)")
print("validation NLL: statistical core %.4f -> SC-LSTM %.4f" % (rr["val_nll_core"].mean(), rr["val_nll"].mean()))

### 8d. Residual correlation, combination, errors by volume group; hybrid figure

In [ ]:
# Residual correlation, combination, errors by volume group
Yp = np.concatenate(pooled["y"]); Pc = np.concatenate(pooled["core"]); Ph = np.concatenate(pooled["hyb"]); have_l = all(p is not None for p in pooled["lstm"])
Pl = np.concatenate(pooled["lstm"]) if have_l else None; lvl_m = np.mean(pooled["lvl"], 0); ne = lvl_m >= EMPTY_THRESH; j = 1   # ride_out
def mae_by_group(y, p):
    sm = np.abs(y[:, :, j] - np.maximum(p[:, :, j], 0)).mean(0); g = np.digitize(lvl_m[:, j], GROUP_EDGES)
    return {"all": sm[ne[:, j]].mean(), **{n: sm[(g == i) & ne[:, j]].mean() for i, n in enumerate(("small", "mid", "large"))}}
print("ride_out MAE by volume group (non-empty stations):"); gt = {"stat_core": mae_by_group(Yp, Pc), "SC-LSTM": mae_by_group(Yp, Ph)}
if have_l: gt.update({"LSTM_full": mae_by_group(Yp, Pl), "ens_equal": mae_by_group(Yp, (Pc + Pl) / 2)})
gt = pd.DataFrame(gt).T; print(gt.round(3).to_string()); gt.to_csv(OUT9 / "mae_by_group.csv")
if have_l:
    e_c, e_l, e_h = ((Yp - P)[:, :, j][:, ne[:, j]].ravel() for P in (Pc, Pl, Ph))
    print(f"\nResidual correlation (station-hour, ride_out): core vs LSTM_full {np.corrcoef(e_c, e_l)[0,1]:.3f} | core vs SC-LSTM {np.corrcoef(e_c, e_h)[0,1]:.3f} | LSTM_full vs SC-LSTM {np.corrcoef(e_l, e_h)[0,1]:.3f}")
    print(f"Forecast correlation: core vs LSTM_full {np.corrcoef(Pc[:, :, j][:, ne[:, j]].ravel(), Pl[:, :, j][:, ne[:, j]].ravel())[0,1]:.3f}")
    print(f"MAE: core {np.abs(e_c).mean():.3f} | LSTM_full {np.abs(e_l).mean():.3f} | SC-LSTM {np.abs(e_h).mean():.3f} | equal-weight average {np.abs(((Yp - (Pc + Pl) / 2))[:, :, j][:, ne[:, j]]).mean():.3f}")
    smc, sml, smh = (np.abs(Yp[:, :, j] - np.maximum(P[:, :, j], 0)).mean(0) for P in (Pc, Pl, Ph)); sme = np.abs(Yp[:, :, j] - (Pc + Pl)[:, :, j] / 2).mean(0)
    for a, b, na, nb_ in ((smh, smc, "SC-LSTM", "stat_core"), (smh, sml, "SC-LSTM", "LSTM_full"), (sme, smc, "ens_equal", "stat_core"), (sme, sml, "ens_equal", "LSTM_full")):
        d = (a - b)[ne[:, j]]; print(f"  {na} − {nb_}: mean {d.mean():+.4f} | {na} better at {100*(d<0).mean():.0f}% | p = {wilcoxon(d).pvalue:.3g}")
else: print("\nLSTM_full predictions not found (lstm_full/pred_LSTM_full_*); skipping residual correlation and combination.")

In [ ]:
# Figure: fig_hybrid_mae
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})
COL = {"stat_core": "#1f3f73", "LSTM_full": "#e07b39", "SC-LSTM": "#c23b3b", "ens_equal": "#5fa052", "ens_flow": "#8ab46a", "ens_hyb_lstm": "#9b59b6"}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
t1 = tab.iloc[::-1]; axes[0].barh(t1.index, t1["ride_out_MAE"], xerr=t1["ride_out_MAE_sd"].fillna(0), color=[COL.get(m, "#888") for m in t1.index], ecolor="#444", capsize=2)
for i, v in enumerate(t1["ride_out_MAE"]): axes[0].text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=7)
axes[0].set_xlim(t1["ride_out_MAE"].min() * 0.9, t1["ride_out_MAE"].max() * 1.06); axes[0].set_xlabel("ride_out MAE per station-hour (non-empty stations, 3 folds)"); axes[0].grid(axis="x", alpha=0.3)
g2 = gt[["small", "mid", "large"]]; wdt = 0.8 / len(g2)
for i, (m, r) in enumerate(g2.iterrows()): axes[1].bar(np.arange(3) + i * wdt, r.to_numpy(), wdt, color=COL.get(m, "#888"), label=m)
axes[1].set_xticks(np.arange(3) + 0.4 - wdt / 2); axes[1].set_xticklabels(["< 0.5 /h", "0.5–2 /h", "≥ 2 /h"]); axes[1].set_ylabel("ride_out MAE"); axes[1].legend(fontsize=7); axes[1].grid(axis="y", alpha=0.3)
axes[1].set_title("by station flow group", fontsize=9, loc="left"); axes[0].set_title("Statistical core, LSTM, hybrid and ensembles", fontsize=9, loc="left")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig_hybrid_mae.png", dpi=300, bbox_inches="tight"); plt.show()

## 9. Network-total forecasts (Tables 7 and 8)

All stations summed into one series: naive references, statistical pipeline, SARIMAX, ridge, gradient boosting and the three deep models.

In [ ]:
# Configuration
import itertools, json, math, time, warnings, pickle
from pathlib import Path
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")

OUT_DIR   = FORECAST_DIR / "network_total"; OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUT_DIR / "results_total.csv"
TUNE_CSV    = OUT_DIR / "tune_runs_total.csv"

BUFFER    = 200
RES_LIST  = ["1h"]
FEATURES  = ["ar", "semi_ar"]
ATTENTION = [False]
DEEP_MODELS = ["LSTM", "ST-GCN", "MBC-GCN"]
SEEDS     = [0, 1, 2]
N_FOLDS, VAL_DAYS, TEST_DAYS = 3, 2, 2
FORCE_CPU = False

MAX_EPOCHS, PATIENCE = 300, 30
WARMUP_EPOCHS, MIN_LR = 5, 1e-5
SEQ_LEN_DEFAULT = {"1h": 8, "15min": 12}
HP = {
    "LSTM":    {"hidden": 32, "layers": 1, "dropout": 0.1, "lr": 2e-3, "wd": 1e-4, "batch": 16},
    "ST-GCN":  {"hidden": 32, "layers": 1, "k": 1, "dropout": 0.1, "lr": 2e-3, "wd": 1e-4, "batch": 16},
    "MBC-GCN": {"hidden": 32, "layers": 1, "k": 1, "dropout": 0.1, "time_dim": 8, "lr": 2e-3, "wd": 1e-4, "batch": 16},
}

TUNE = True
N_TRIALS = {"LSTM": 25, "ST-GCN": 25, "MBC-GCN": 25, "HGB": 40}
TUNE_FOLDS, TUNE_SEEDS = [0, 1, 2], [0, 1]
TUNE_FEATURES, TUNE_ATTENTION = "semi_ar", False
SPACE_COMMON = {"hidden": ("cat", [16, 32, 64, 128]), "layers": ("cat", [1, 2, 3]), "dropout": ("float", 0.0, 0.5),
                "lr": ("log", 3e-4, 1e-2), "wd": ("log", 1e-7, 1e-2), "batch": ("cat", [8, 16, 32])}
SPACE_SEQ = {"1h": ("cat", [4, 8, 12, 18]), "15min": ("cat", [8, 12, 24, 36])}
SPACE_HGB = {"learning_rate": ("log", 0.01, 0.3), "max_iter": ("cat", [100, 200, 400, 800]), "max_leaf_nodes": ("cat", [4, 8, 16, 31]),
             "min_samples_leaf": ("cat", [5, 10, 20, 40]), "l2_regularization": ("log", 1e-3, 10.0)}
HP_HGB = {"learning_rate": 0.05, "max_iter": 300, "max_leaf_nodes": 8, "min_samples_leaf": 10, "l2_regularization": 0.1}

RIDGE_LAMS = (0.01, 0.1, 1, 10, 100, 1000)
PROF_GRID  = {"half_life": [1, 2, 3, 5, np.inf],
              "dshrink": [0, 1, 2, 4, 8],
              "ratio_prior": [0, 500, 2000, 5000],
              "beta": [0.0, 0.25, 0.5, 0.75, 1.0]}
SHORT_H = 3
USE_SARIMA = True
SARIMA_ORDER, SARIMA_SEASONAL = (2, 0, 1), (1, 1, 0)

In [ ]:
# Data: network-total series
device = torch.device("cpu") if FORCE_CPU else get_device(); print("device:", device)
L1 = torch.ones(1, 1)
data = {}
for res in RES_LIST:
    p = load_panel(PANEL_DIR / f"panel_{res}_{BUFFER}m.csv"); X, w, nodes = panel_to_tensor(p)
    Xt = np.concatenate([X[:, :, :4].sum(1, keepdims=True), X[:, :1, 4:]], 2).astype(np.float32)
    data[res] = dict(X=Xt, w=w, n_stations=X.shape[1], folds=day_folds(w, VAL_DAYS, TEST_DAYS, N_FOLDS))
    print(f"[{res}] {X.shape[1]} stations -> total series {Xt.shape} | mean per step ride_in {Xt[:,0,0].mean():.0f}, ride_out {Xt[:,0,1].mean():.0f}, "
          f"metro in {Xt[:,0,2].mean():.0f}, out {Xt[:,0,3].mean():.0f} | bike total min/max {Xt[:,0,1].min():.0f}/{Xt[:,0,1].max():.0f}")
    for k, f in enumerate(data[res]["folds"]):
        print(f"   fold {k+1}: train {f['train'][0]}–{f['train'][-1]} ({len(f['train'])} d) | val {f['val'][0]}–{f['val'][-1]} | test {f['test'][0]}–{f['test'][-1]}")
    del X, p

In [ ]:
# Functions: targets, naive references, statistical pipeline, ridge, gradient boosting, SARIMA, evaluation
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
from scipy.stats import nbinom, wilcoxon
try:
    from sklearn.ensemble import HistGradientBoostingRegressor; HAVE_SK = True
except ImportError: HAVE_SK = False; print("scikit-learn not installed; skipping gradient boosting (pip install scikit-learn)")
try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING); HAVE_OPTUNA = True
except ImportError: HAVE_OPTUNA = False; print("optuna not installed; falling back to random search (pip install optuna)")
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX; HAVE_SM = True
except ImportError: HAVE_SM = False; print("statsmodels not installed; skipping SARIMA (pip install statsmodels)")

def series_ctx(res):
    D = data[res]; X, w = D["X"], D["w"]; T = len(X)
    dates = pd.to_datetime(pd.Series(np.asarray(w.date))).dt.normalize().to_numpy(); udays = np.unique(dates)
    day_id = np.searchsorted(udays, dates); first_t = np.searchsorted(day_id, np.arange(len(udays)))
    SPD = int((day_id == 0).sum()); assert (np.bincount(day_id) == SPD).all(), "unequal number of steps per day"
    return dict(res=res, X=X, w=w, T=T, Y=X[:, 0, :2].astype(float), M=X[:, 0, 2:4].astype(float), TF=X[:, 0, 4:].astype(float),
                dates=dates, udays=udays, day_id=day_id, slot=np.arange(T) - first_t[day_id], SPD=SPD, n_days=len(udays),
                is_wk=(pd.DatetimeIndex(udays).weekday >= 5).astype(int), S=SEQ_LEN_DEFAULT[res])

def targets_of(idx, T, S=1):
    """Target times t+1 with t and t+1 in the segment and a full input window; shared by all models."""
    s = set(int(i) for i in idx); return np.array(sorted(t + 1 for t in idx if (t + 1) in s and t - S + 1 >= 0 and t + 1 < T), dtype=int)

def naive_preds(c, sp, tg):
    Y, SPD = c["Y"], c["SPD"]; tr = np.asarray(sp["train"])
    key_tr = c["slot"][tr] * 2 + c["is_wk"][c["day_id"][tr]]; hist = {k: Y[tr[key_tr == k]].mean(0) for k in np.unique(key_tr)}
    key_tg = c["slot"][tg] * 2 + c["is_wk"][c["day_id"][tg]]
    return {"persistence": Y[tg - 1], "seasonal_naive": np.where((tg - SPD >= 0)[:, None], Y[np.maximum(tg - SPD, 0)], Y[tg - 1]),
            "hist_mean": np.stack([hist.get(k, Y[tr].mean(0)) for k in key_tg])}

def build_profile(c, hl, ds):
    """Causal intraday profile: day d uses earlier days only, weights 0.5^((d-i-1)/hl); day-type profile shrunk towards the all-day profile."""
    nd, SPD = c["n_days"], c["SPD"]; Yd = c["Y"].reshape(nd, SPD, 2); i = np.arange(nd); lag = i[:, None] - i[None, :] - 1
    wgt = np.where(lag >= 0, (0.5 ** (np.maximum(lag, 0) / hl)) if np.isfinite(hl) else 1.0, 0.0)
    same = c["is_wk"][:, None] == c["is_wk"][None, :]; Wdt = wgt * same
    raw_all = np.einsum("di,ihc->dhc", wgt, Yd) / np.maximum(wgt.sum(1), 1e-12)[:, None, None]
    raw_dt = np.einsum("di,ihc->dhc", Wdt, Yd) / np.maximum(Wdt.sum(1), 1e-12)[:, None, None]
    Wd = Wdt.sum(1); den = (Wd + ds)[:, None, None]
    prof = np.where(den > 0, (Wd[:, None, None] * raw_dt + ds * raw_all) / np.maximum(den, 1e-12), raw_all)
    prof[0] = Yd[0]
    return prof.reshape(-1, 2)

def _winsum(a, L):
    cs = np.cumsum(np.concatenate([np.zeros((1,) + a.shape[1:]), a], 0), 0); t = np.arange(len(a)); out = np.full_like(a, np.nan)
    ok = t - L >= 0; out[ok] = cs[t[ok]] - cs[t[ok] - L]; return out    # Σ_{t−L..t−1}

def level_ratio(c, B, prior, beta):
    """Recent-level ratio: previous service day and previous SHORT_H steps, combined geometrically."""
    Y, SPD = c["Y"], c["SPD"]; rd = (_winsum(Y, SPD) + prior) / (_winsum(B, SPD) + prior)
    ps = prior * SHORT_H / SPD; rs = (_winsum(Y, SHORT_H) + ps) / (_winsum(B, SHORT_H) + ps)
    r = np.nan_to_num(rd, nan=1.0) ** (1 - beta) * np.nan_to_num(rs, nan=1.0) ** beta
    return np.where(np.isfinite(r), r, 1.0)

def select_profile(c, sp):
    """Joint grid search of the profile parameters on training and validation days (sum of the two MAEs)."""
    tv = np.r_[targets_of(sp["train"], c["T"]), targets_of(sp["val"], c["T"])]; tv = tv[c["day_id"][tv] >= 2]
    best, rows = None, []
    for hl, ds in itertools.product(PROF_GRID["half_life"], PROF_GRID["dshrink"]):
        B = build_profile(c, hl, ds); mae_raw = np.abs(c["Y"][tv] - B[tv]).mean()
        for pr, be in itertools.product(PROF_GRID["ratio_prior"], PROF_GRID["beta"]):
            mu = B * level_ratio(c, B, pr, be); mae = np.abs(c["Y"][tv] - mu[tv]).mean()
            rows.append(dict(half_life=hl, dshrink=ds, ratio_prior=pr, beta=be, mae_raw=mae_raw, mae=mae))
            if best is None or mae < best[0]: best = (mae, dict(half_life=hl, dshrink=ds, ratio_prior=pr, beta=be))
    g = pd.DataFrame(rows); raw_best = g.loc[g["mae_raw"].idxmin(), ["half_life", "dshrink"]].to_dict()
    return best[1], raw_best, g

def fit_phi(y, mu):
    y = y.ravel(); mu = np.maximum(mu.ravel(), 1e-3)
    def nll(lp):
        phi = math.exp(lp); return -np.mean(gammaln(y + phi) - gammaln(phi) - gammaln(y + 1) + phi * np.log(phi / (phi + mu)) + y * np.log(mu / (phi + mu)))
    return math.exp(minimize_scalar(nll, bounds=(-2, 12), method="bounded").x)

def design(c, tg, S, semi):
    Z = np.log1p(np.c_[c["Y"], c["M"]]); cols = [0, 1, 2, 3] if semi else [0, 1]; SPD = c["SPD"]
    lags = np.stack([Z[tg - 1 - l][:, cols] for l in range(S)], 1).reshape(len(tg), -1)
    yday = np.where((tg - SPD >= 0)[:, None], Z[np.maximum(tg - SPD, 0)][:, :2], Z[tg - 1][:, :2])
    return np.c_[lags, yday, c["TF"][tg], c["is_wk"][c["day_id"][tg]]]

def ridge_model(c, sp, semi, S, lams=RIDGE_LAMS):
    """Ridge regression on the log scale; lambda chosen on validation MAE; smearing correction."""
    T = c["T"]; tg = {k: targets_of(sp[k], T, S) for k in ("train", "val", "test")}
    F_ = {k: design(c, v, S, semi) for k, v in tg.items()}; Ylog = {k: np.log1p(c["Y"][v]) for k, v in tg.items()}
    mu, sd = F_["train"].mean(0), F_["train"].std(0) + 1e-9; norm = lambda A: np.c_[(A - mu) / sd, np.ones(len(A))]
    def fit(lam):
        A = norm(F_["train"]); P = lam * np.eye(A.shape[1]); P[-1, -1] = 0; return np.linalg.solve(A.T @ A + P, A.T @ Ylog["train"])
    def pred(W, k): sm = np.exp(Ylog["train"] - norm(F_["train"]) @ W).mean(0); return (np.exp(norm(F_[k]) @ W) * sm - 1).clip(0)
    lam = min(lams, key=lambda l: np.abs(pred(fit(l), "val") - c["Y"][tg["val"]]).mean()); W = fit(lam)
    return pred(W, "test"), pred(W, "val"), lam

def hgb_model(c, sp, semi, S, hp, seed=0):
    """HistGradientBoosting on log1p targets, one model per direction; fitted on training days."""
    T = c["T"]; tg = {k: targets_of(sp[k], T, S) for k in ("train", "val", "test")}; F_ = {k: design(c, v, S, semi) for k, v in tg.items()}
    out = {k: np.zeros((len(v), 2)) for k, v in tg.items()}
    for j in range(2):
        m = HistGradientBoostingRegressor(random_state=seed, early_stopping=False, **hp).fit(F_["train"], np.log1p(c["Y"][tg["train"], j]))
        for k in out: out[k][:, j] = np.expm1(m.predict(F_[k])).clip(0)
    return out["test"], out["val"]

def sarima_model(c, sp, semi, tg_te):
    """SARIMA(X) on the log1p scale, seasonal period = steps per day; one-step-ahead forecasts on test days without refitting."""
    T, SPD = c["T"], c["SPD"]; fit_end = int(max(sp["val"])) + 1; Z = np.log1p(c["Y"]); ex = np.log1p(np.r_[c["M"][:1], c["M"][:-1]]) if semi else None
    out = np.zeros((len(tg_te), 2))
    for j in range(2):
        try:
            mod = SARIMAX(Z[:fit_end, j], exog=None if ex is None else ex[:fit_end], order=SARIMA_ORDER, seasonal_order=SARIMA_SEASONAL + (SPD,),
                          enforce_stationarity=False, enforce_invertibility=False)
            res = mod.fit(disp=False, maxiter=200)
            full = res.apply(Z[:, j], exog=None if ex is None else ex)
            out[:, j] = np.expm1(full.get_prediction(start=int(tg_te[0]), end=int(tg_te[-1])).predicted_mean)[tg_te - tg_te[0]].clip(0)
        except Exception as e:
            print(f"   SARIMA failed ({'semi' if semi else 'ar'}, dir {j}): {e}"); out[:, j] = np.nan
    return out

def eval_point(y, p):
    out = {}
    for j, tgt in enumerate(("ride_in", "ride_out")):
        yt, pt = y[:, j], np.maximum(p[:, j], 0); e = yt - pt
        out[f"{tgt}_MAE"] = np.abs(e).mean(); out[f"{tgt}_RMSE"] = np.sqrt((e ** 2).mean()); out[f"{tgt}_MAPE"] = 100 * np.abs(e / np.maximum(yt, 1)).mean()
        out[f"{tgt}_R2"] = 1 - (e ** 2).sum() / ((yt - yt.mean()) ** 2).sum(); out[f"{tgt}_bias_ratio"] = pt.sum() / yt.sum()
    return out

In [ ]:
# Functions: deep-model run and tuning
def feat_cols(features):
    flows = [0, 1] + ([2, 3] if features == "semi_ar" else []); return flows, list(range(4, 4 + len(TIME_FEATS)))

def hp_key(hp): return json.dumps({k: (round(v, 6) if isinstance(v, float) else v) for k, v in hp.items()}, sort_keys=True)
KEY = ["res", "fold", "features", "attention", "model", "seed"]

def run_deep(res, fold_id, features, attention, model_name, seed, hp, final=True):
    """Train one configuration; returns (row, test predictions, validation predictions)."""
    D = data[res]; X, w = D["X"], D["w"]; sp = split_index(w, D["folds"][fold_id]); sc = Scaler().fit(X, sp["train"]); Xs = sc.transform(X)
    flows, tcols = feat_cols(features); seq = hp.get("seq_len", SEQ_LEN_DEFAULT[res])
    tr, va, te = make_loaders(Xs, sp, seq, flows + tcols, hp["batch"])
    def build():
        set_seed(seed); return build_model(model_name, 1, len(flows), len(tcols), hp, attention)
    t0 = time.time(); m, best_val, epochs, best_ep, hit_cap, dev_used = train_safe(build, tr, va, L1, device, hp["lr"], hp["wd"], MAX_EPOCHS, PATIENCE)
    yv, pv = predict(m, va, L1, dev_used); yv, pv = sc.inverse_targets(yv)[:, 0], sc.inverse_targets(pv)[:, 0]
    row = {"res": res, "fold": fold_id + 1, "features": features, "attention": attention, "model": model_name, "seed": seed, "hp": hp_key(hp),
           "val_mse": best_val, "val_MAE": float(np.abs(yv - pv).mean()), "epochs": epochs, "best_ep": best_ep, "hit_cap": hit_cap, "device": dev_used}
    p_te = None
    if final:
        yt, pt = predict(m, te, L1, dev_used); yt, p_te = sc.inverse_targets(yt)[:, 0], sc.inverse_targets(pt)[:, 0]
        c = series_ctx(res); tg = targets_of(sp["test"], c["T"], seq); assert len(tg) == len(yt) and np.allclose(c["Y"][tg], yt, atol=1.0), "deep-model test targets do not match targets_of"
        row.update(eval_point(yt, p_te))
    row["secs"] = round(time.time() - t0); return row, p_te, pv

def append_csv(path, df): df.to_csv(path, mode="a", header=not path.exists(), index=False)

tune_cache = {}
if TUNE_CSV.exists():
    for _, r in pd.read_csv(TUNE_CSV).iterrows(): tune_cache[(r["res"], r["model"], r["hp"], int(r["fold"]), int(r["seed"]))] = r.to_dict()

def draw(spec, trial=None, rng=None, name=""):
    kind = spec[0]
    if trial is not None:
        return trial.suggest_categorical(name, spec[1]) if kind == "cat" else trial.suggest_float(name, spec[1], spec[2], log=(kind == "log"))
    if kind == "cat": return spec[1][int(rng.integers(len(spec[1])))]
    return float(np.exp(rng.uniform(np.log(spec[1]), np.log(spec[2])))) if kind == "log" else float(rng.uniform(spec[1], spec[2]))

def clean(hp):
    for k in ("hidden", "layers", "batch", "seq_len", "k", "max_iter", "max_leaf_nodes", "min_samples_leaf"):
        if k in hp: hp[k] = int(hp[k])
    return hp

def diff_label(hp, base): return ", ".join(f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in hp.items() if base.get(k) != v) or "default"

def tune_generic(name, base, space, n_trials, evaluate_hp, seeds):
    """TPE search: evaluate_hp(hp, fold, seed) -> validation MAE; trial 0 is the default; returns the best hp."""
    print(f"[tune] {name}: {n_trials} trials x {len(TUNE_FOLDS)} folds x {len(seeds)} seeds"); T0 = time.time(); rows = []
    def evaluate(hp, i, label):
        vals = np.array([[evaluate_hp(hp, f, s) for s in seeds] for f in TUNE_FOLDS]); val = float(vals.mean())
        rows.append({"trial": i, "val": val, "seed_spread": float((vals.max(1) - vals.min(1)).mean()), "label": label, **hp})
        print(f"  [{i:>2}] val MAE {val:.2f} | {label}"); return val
    if HAVE_OPTUNA:
        pkl = OUT_DIR / f"optuna_total_{name}.pkl"
        study = pickle.load(open(pkl, "rb")) if pkl.exists() else optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0, n_startup_trials=max(5, n_trials // 4)))
        if len(study.trials) == 0:
            study.enqueue_trial({k: base[k] for k, spec in space.items() if ((base[k] in spec[1]) if spec[0] == "cat" else (spec[1] <= base[k] <= spec[2]))})
        def objective(trial):
            hp = clean({**base, **{k: draw(spec, trial, name=k) for k, spec in space.items()}}); return evaluate(hp, trial.number, diff_label(hp, base))
        n_done = len([t for t in study.trials if t.state.is_finished()])
        study.optimize(objective, n_trials=max(0, n_trials - n_done), callbacks=[lambda st, tr: pickle.dump(st, open(pkl, "wb"))])
        if not rows:
            for t in study.trials: evaluate(clean({**base, **t.params}), t.number, diff_label(clean({**base, **t.params}), base))
        best = clean({**base, **study.best_trial.params})
        try: print("  parameter importance:", {k: round(float(v), 3) for k, v in optuna.importance.get_param_importances(study).items()})
        except Exception: pass
    else:
        rng = np.random.default_rng(0); cands = [dict(base)] + [clean({**base, **{k: draw(spec, rng=rng) for k, spec in space.items()}}) for _ in range(n_trials - 1)]
        for i, hp in enumerate(cands): evaluate(hp, i, diff_label(hp, base))
        best = cands[int(np.argmin([r["val"] for r in rows]))]
    tab = pd.DataFrame(rows).sort_values("val").reset_index(drop=True); top = tab["val"].iloc[:5]; spread = tab["seed_spread"].iloc[0]
    print(f"  best {top.iloc[0]:.2f} vs default {tab[tab.trial == 0]['val'].iloc[0]:.2f} | gap best→5th {top.iloc[-1] - top.iloc[0]:.2f} vs seed spread {spread:.2f}"
          + ("  -> top trials differ by less than the seed spread: search is saturated" if top.iloc[-1] - top.iloc[0] < spread else "  -> trials still differ: N_TRIALS could be increased"))
    for k, spec in space.items():
        if spec[0] != "cat":
            lo, hi, v = spec[1], spec[2], best[k]; pos = (np.log(v / lo) / np.log(hi / lo)) if spec[0] == "log" else (v - lo) / (hi - lo)
            if pos < 0.05 or pos > 0.95: print(f"  ! {k}={v:.3g} is close to the search bound [{lo}, {hi}]; consider widening it")
    print(f"  took {(time.time()-T0)/60:.1f} min"); tab.to_csv(OUT_DIR / f"tune_table_total_{name}.csv", index=False); return best

def deep_val_mae(res, model_name):
    def ev(hp, f, s):
        key = (res, model_name, hp_key(hp), f + 1, s)
        if key not in tune_cache:
            row, _, _ = run_deep(res, f, TUNE_FEATURES, TUNE_ATTENTION, model_name, s, hp, final=False); tune_cache[key] = row; append_csv(TUNE_CSV, pd.DataFrame([row]))
        return float(tune_cache[key]["val_MAE"])
    return ev

HP_FINAL = {res: {m: {**HP[m], "seq_len": SEQ_LEN_DEFAULT[res]} for m in DEEP_MODELS} for res in RES_LIST}
if TUNE:
    for res in RES_LIST:
        for m in DEEP_MODELS:
            HP_FINAL[res][m] = tune_generic(f"{res}_{m}", {**HP[m], "seq_len": SEQ_LEN_DEFAULT[res]}, {**SPACE_COMMON, "seq_len": SPACE_SEQ[res]}, N_TRIALS[m], deep_val_mae(res, m), TUNE_SEEDS)
json.dump(HP_FINAL, open(OUT_DIR / "hp_final_total.json", "w"), indent=2)
print("\nfinal deep hyper-parameters:")
for res in RES_LIST: print(f"[{res}]"); print(pd.DataFrame.from_dict(HP_FINAL[res], orient="index").round(5).to_string())

In [ ]:
# Gradient-boosting tuning
HP_HGB_FINAL = {res: dict(HP_HGB) for res in RES_LIST}
if HAVE_SK and TUNE:
    for res in RES_LIST:
        c = series_ctx(res)
        def ev(hp, f, s, c=c, res=res):
            sp = split_index(c["w"], data[res]["folds"][f]); _, pv = hgb_model(c, sp, TUNE_FEATURES == "semi_ar", c["S"], hp, seed=s)
            return float(np.abs(pv - c["Y"][targets_of(sp["val"], c["T"], c["S"])]).mean())
        HP_HGB_FINAL[res] = tune_generic(f"{res}_HGB", dict(HP_HGB), SPACE_HGB, N_TRIALS["HGB"], ev, [0])
json.dump(HP_HGB_FINAL, open(OUT_DIR / "hp_final_hgb.json", "w"), indent=2); print(HP_HGB_FINAL)

In [ ]:
# Main loop (resumable)
def load_deep_cache():
    if not RESULTS_CSV.exists(): return {}
    old = pd.read_csv(RESULTS_CSV); old = old[np.isfinite(old["ride_out_MAE"])]; old["attention"] = old["attention"].astype(str).str.lower().eq("true")
    return {tuple(r[k] for k in KEY): r.to_dict() for _, r in old.iterrows() if r["hp"] == hp_key(HP_FINAL[r["res"]][r["model"]])}
def pred_path(res, f, feat, att, mdl, sd): return OUT_DIR / f"pred_{res}_f{f+1}_{feat}_{int(att)}_{mdl}_s{sd}.npy"

FAMILY = {"persistence": "naive", "seasonal_naive": "naive", "hist_mean": "naive", "profile_raw": "statistical", "profile_ratio": "statistical",
          "ridge_ar": "ridge", "ridge_semi_ar": "ridge", "hgb_ar": "hgb", "hgb_semi_ar": "hgb", "sarima_ar": "sarima", "sarima_semi_ar": "sarima"}
rows, curves, params_log = [], {}, {}; deep_cache = load_deep_cache(); print(f"{len(deep_cache)} deep runs cached (matching current hyper-parameters)"); T0 = time.time()
for res in RES_LIST:
    c = series_ctx(res); S = c["S"]
    for f in range(N_FOLDS):
        sp = split_index(c["w"], data[res]["folds"][f]); tg = targets_of(sp["test"], c["T"], S); y = c["Y"][tg]; preds, extra = {}, {}
        preds.update(naive_preds(c, sp, tg))
        bp, raw_bp, grid = select_profile(c, sp); grid.to_csv(OUT_DIR / f"profile_grid_{res}_fold{f+1}.csv", index=False)
        preds["profile_raw"] = build_profile(c, raw_bp["half_life"], raw_bp["dshrink"])[tg]
        B = build_profile(c, bp["half_life"], bp["dshrink"]); MU = B * level_ratio(c, B, bp["ratio_prior"], bp["beta"]); preds["profile_ratio"] = MU[tg]
        tv = np.r_[targets_of(sp["train"], c["T"]), targets_of(sp["val"], c["T"])]; tv = tv[c["day_id"][tv] >= 2]
        phi = np.array([fit_phi(c["Y"][tv, j], MU[tv, j]) for j in range(2)])
        lo, hi = nbinom.ppf(0.05, phi, phi / (phi + np.maximum(MU[tg], 1e-3))), nbinom.ppf(0.95, phi, phi / (phi + np.maximum(MU[tg], 1e-3)))
        extra["profile_ratio"] = {"ride_in_cov90": float(((y[:, 0] >= lo[:, 0]) & (y[:, 0] <= hi[:, 0])).mean()), "ride_out_cov90": float(((y[:, 1] >= lo[:, 1]) & (y[:, 1] <= hi[:, 1])).mean())}
        params_log[(res, f + 1)] = {"profile": bp, "profile_raw": raw_bp, "phi": phi.round(1).tolist()}
        for semi, tag in ((False, "ar"), (True, "semi_ar")):
            p_te, _, lam = ridge_model(c, sp, semi, S); preds[f"ridge_{tag}"] = p_te; params_log[(res, f + 1)][f"ridge_{tag}_lambda"] = lam
            if HAVE_SK: preds[f"hgb_{tag}"], _ = hgb_model(c, sp, semi, S, HP_HGB_FINAL[res])
            if USE_SARIMA and HAVE_SM:
                p_sa = sarima_model(c, sp, semi, tg)
                if np.isfinite(p_sa).all(): preds["sarima_ar" if not semi else "sarima_semi_ar"] = p_sa
        for name, p in preds.items():
            rows.append({"res": res, "fold": f + 1, "model": name, "family": FAMILY[name], "features": "semi_ar" if "semi" in name else ("ar" if name.split("_")[-1] == "ar" else "-"),
                         "attention": False, "seed": 0, **eval_point(y, p), **extra.get(name, {})})
        for feat, att, mdl in itertools.product(FEATURES, ATTENTION, DEEP_MODELS):
            ps = []
            for sd in SEEDS:
                key = (res, f + 1, feat, att, mdl, sd); pp = pred_path(res, f, feat, att, mdl, sd)
                if key in deep_cache and pp.exists(): row = deep_cache[key]; p_te = np.load(pp)
                else:
                    row, p_te, _ = run_deep(res, f, feat, att, mdl, sd, HP_FINAL[res][mdl]); np.save(pp, p_te); append_csv(RESULTS_CSV, pd.DataFrame([row])); deep_cache[key] = row
                    print(f"   {res} f{f+1} {mdl:8s} {feat:8s} attn={att!s:5s} s{sd} | MAE in/out {row['ride_in_MAE']:.1f}/{row['ride_out_MAE']:.1f} | ep {row['epochs']} (best {row['best_ep']})"
                          + ("  [hit epoch cap]" if row["hit_cap"] else "") + f" | {row['secs']}s | {(time.time()-T0)/60:.1f} min")
                assert len(p_te) == len(tg), "deep models and other models have different numbers of test targets"
                rows.append({**{k: row[k] for k in ["res", "fold", "features", "attention", "model", "seed", "epochs", "best_ep", "hit_cap"]}, "family": "deep", **eval_point(y, p_te)}); ps.append(p_te)
            preds[f"{mdl} {feat}{'+attn' if att else ''}"] = np.mean(ps, 0)
        curves[(res, f + 1)] = {"tg": tg, "y": y, "preds": preds}
        np.savez_compressed(OUT_DIR / f"preds_total_{res}_fold{f+1}.npz", tg=tg, y=y, **{k.replace(" ", "_"): v for k, v in preds.items()})
        print(f"[{res}] fold {f+1} done | test days {pd.DatetimeIndex(c['udays'][np.unique(c['day_id'][tg])]).strftime('%m-%d').tolist()} | profile parameters {bp} | φ {phi.round(1).tolist()} | {(time.time()-T0)/60:.1f} min")
res_df = pd.DataFrame(rows); res_df.to_csv(OUT_DIR / "metrics_total_all_runs.csv", index=False)
json.dump({f"{k[0]}_fold{k[1]}": v for k, v in params_log.items()}, open(OUT_DIR / "params_total.json", "w"), indent=2, default=float)
conv = res_df[res_df["family"] == "deep"].groupby(["res", "model"]).agg(runs=("epochs", "size"), epochs=("epochs", "mean"), best_ep=("best_ep", "mean"), hit_cap=("hit_cap", lambda x: int(x.astype(bool).sum()))).round(1)
print(conv.to_string()); print(f"{int(conv['hit_cap'].sum())} runs hit MAX_EPOCHS={MAX_EPOCHS} while still improving")

In [ ]:
# Summary table, metro contribution, paired tests
METRICS = ["ride_in_MAE", "ride_out_MAE", "ride_in_RMSE", "ride_out_RMSE", "ride_in_MAPE", "ride_out_MAPE", "ride_in_R2", "ride_out_R2", "ride_out_bias_ratio"]
res_df["label"] = np.where(res_df["family"] == "deep", res_df["model"] + " " + res_df["features"] + np.where(res_df["attention"].astype(bool), "+attn", ""), res_df["model"])
FAM_ORDER = ["naive", "statistical", "sarima", "ridge", "hgb", "deep"]
summary = {}
for res in RES_LIST:
    d = res_df[res_df["res"] == res]
    g = d.groupby(["family", "label"]); tab = g[METRICS].mean(); tab.columns = [c_ + "_mean" for c_ in tab.columns]
    sd = g[["ride_in_MAE", "ride_out_MAE"]].std(ddof=0); tab["ride_in_MAE_sd"] = sd["ride_in_MAE"]; tab["ride_out_MAE_sd"] = sd["ride_out_MAE"]
    tab["runs"] = g.size(); tab = tab.reset_index(); tab["fo"] = tab["family"].map({f_: i for i, f_ in enumerate(FAM_ORDER)})
    hist = tab.loc[tab["label"] == "hist_mean"]; tab["vs_hist_%"] = 100 * (tab["ride_out_MAE_mean"] / float(hist["ride_out_MAE_mean"].iloc[0]) - 1)
    if "ride_out_cov90" in d.columns: tab = tab.merge(d.groupby("label")["ride_out_cov90"].mean().rename("cov90_out"), left_on="label", right_index=True, how="left")
    tab = tab.sort_values(["fo", "ride_out_MAE_mean"]).drop(columns="fo").reset_index(drop=True); summary[res] = tab; tab.to_csv(OUT_DIR / f"summary_total_{res}.csv", index=False)
    mean_tot = res_df[res_df["res"] == res]; c = series_ctx(res)
    print(f"\n===== {res} × {BUFFER} m | network-total hourly forecasts (3 folds; deep models x 3 seeds) | mean test total ride_in {np.mean([v['y'][:,0].mean() for k, v in curves.items() if k[0]==res]):.0f} / ride_out {np.mean([v['y'][:,1].mean() for k, v in curves.items() if k[0]==res]):.0f} /h =====")
    show = ["family", "label", "runs", "ride_in_MAE_mean", "ride_out_MAE_mean", "ride_out_MAE_sd", "ride_in_RMSE_mean", "ride_out_RMSE_mean", "ride_out_MAPE_mean", "ride_in_R2_mean", "ride_out_R2_mean", "ride_out_bias_ratio_mean", "vs_hist_%"]
    print(tab[show].round(2).to_string(index=False))
    best = tab.loc[tab.groupby("family")["ride_out_MAE_mean"].idxmin()].sort_values("ride_out_MAE_mean")
    print("\nBest model of each family (by ride_out MAE):"); print(best[["family", "label", "ride_in_MAE_mean", "ride_out_MAE_mean", "ride_out_R2_mean", "vs_hist_%"]].round(2).to_string(index=False))

    d2 = d[d["features"].isin(["ar", "semi_ar"])].copy(); d2["base"] = d2["label"].str.replace("_semi_ar", "_ar").str.replace(" semi_ar", " ar")
    pv = d2.pivot_table(index=["base", "fold", "seed"], columns="features", values=["ride_in_MAE", "ride_out_MAE"])
    gain = pd.DataFrame({"ride_in_%": 100 * (1 - pv[("ride_in_MAE", "semi_ar")] / pv[("ride_in_MAE", "ar")]), "ride_out_%": 100 * (1 - pv[("ride_out_MAE", "semi_ar")] / pv[("ride_out_MAE", "ar")])}).reset_index()
    gs = gain.groupby("base")[["ride_in_%", "ride_out_%"]].agg(["mean", "std", "count"]).round(2)
    pvals = {}
    for b_, gg in gain.groupby("base"):
        dd = gg["ride_out_%"].to_numpy(); pvals[b_] = wilcoxon(dd, alternative="greater").pvalue if len(dd) >= 5 and (dd != 0).any() else np.nan
    gs[("ride_out_%", "p_greater")] = pd.Series(pvals); print("\nGain from metro totals (positive = Semi-AR better; one-sided Wilcoxon, paired by fold x seed):"); print(gs.to_string())
    ref = "profile_ratio"; cv = [v for k, v in curves.items() if k[0] == res]; yy = np.concatenate([v["y"][:, 1] for v in cv]); e_ref = np.abs(yy - np.concatenate([v["preds"][ref][:, 1] for v in cv]))
    out = []
    for name in cv[0]["preds"]:
        if name == ref: continue
        e = np.abs(yy - np.concatenate([v["preds"][name][:, 1] for v in cv])); dlt = e - e_ref
        out.append({"model": name, "MAE": e.mean(), "ref_MAE": e_ref.mean(), "diff": dlt.mean(), "ref_better_%": 100 * (dlt > 0).mean(), "p_two_sided": wilcoxon(dlt).pvalue if (dlt != 0).any() else np.nan})
    pt = pd.DataFrame(out).sort_values("MAE"); print(f"\nPaired by hour (ride_out, {len(yy)} test hours): each model vs the statistical reference {ref} (diff > 0 = reference better)"); print(pt.round(3).to_string(index=False))
    pt.to_csv(OUT_DIR / f"paired_vs_{ref}_{res}.csv", index=False)
    print("\nParameters by fold:"); print(pd.DataFrame({f"fold{k[1]}": v for k, v in params_log.items() if k[0] == res}).to_string())

In [ ]:
# Save results for Section 12
pickle.dump({"res_df": res_df, "summary": summary, "curves": curves, "params_log": params_log, "HP_FINAL": HP_FINAL, "HP_HGB_FINAL": HP_HGB_FINAL},
            open(OUT_DIR / "session_results.pkl", "wb"))

## 10. 1-km nearest-station panel (protocol of Li et al., 2026)

In [ ]:
# Configuration
import re, glob, json, time, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
warnings.filterwarnings("ignore")

STATION_SOURCES = [str(MERGED_DIR / "merged_2022*.csv"), str(PANEL_DIR / "panel_1h_200m.csv")]

CATCHMENT_M  = 1000
ASSIGN       = "nearest"
DIST_METRIC  = "euclidean"
EVENT_DEF    = "all"
MIN_SEGMENT_SECONDS = 10
BBOX         = (120.0, 122.0, 30.0, 32.0)
RES_LIST     = [5, 15, 30, 60]
DATE_START, DATE_END = "2022-12-12", "2023-01-01"
REF_RADII    = (200, 500, 1000)
OUT_DIR  = PANEL_1KM_DIR / f"{DIST_METRIC}_{EVENT_DEF}"; OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Station table and projection
def load_stations():
    for src in STATION_SOURCES:
        files = sorted(glob.glob(src))
        if not files: continue
        parts = []
        for f in files:
            cols = pd.read_csv(f, nrows=0).columns
            name = next((c for c in cols if c in ("station__stationnames", "stationnames", "station")), None)
            lon  = next((c for c in cols if c in ("station__lon", "lon")), None); lat = next((c for c in cols if c in ("station__lat", "lat")), None)
            sid  = next((c for c in cols if c in ("station__FSTR_STATIONID", "FSTR_STATIONID")), None)
            if not (name and lon and lat): continue
            use = [name, lon, lat] + ([sid] if sid else [])
            t = pd.read_csv(f, usecols=use).drop_duplicates().rename(columns={name: "stationnames", lon: "lon", lat: "lat", **({sid: "sid"} if sid else {})})
            parts.append(t)
        if not parts: continue
        t = pd.concat(parts).dropna(subset=["stationnames", "lon", "lat"])
        if "sid" not in t: t["sid"] = np.nan
        ids = t.dropna(subset=["sid"]).assign(sid=lambda d: d["sid"].astype(int)).groupby("stationnames")["sid"].apply(lambda s: sorted(set(s)))
        st = t.groupby("stationnames", as_index=False)[["lon", "lat"]].mean()
        st["sids"] = st["stationnames"].map(ids).apply(lambda x: x if isinstance(x, list) else [])
        print(f"station table from {src}: {len(st)} station names, {sum(len(x) for x in st['sids'])} station IDs, {len(files)} files")
        return st.reset_index(drop=True)
    raise FileNotFoundError("no file in STATION_SOURCES has station name / lon / lat columns")

LAT0 = None
def to_xy_m(lon, lat):
    """Longitude/latitude -> local equirectangular projection in true metres."""
    R = 6378137.0; return np.c_[R * np.cos(np.radians(LAT0)) * np.radians(np.asarray(lon, float)), R * np.radians(np.asarray(lat, float))]

stations = load_stations(); N = len(stations); LAT0 = float(stations["lat"].mean())
st_xy = to_xy_m(stations["lon"], stations["lat"]); tree = cKDTree(st_xy)
stations.to_csv(OUT_DIR / "stations.csv", index=False)

In [ ]:
# Raw log -> events
RAW_COLS = {"bike": "BICYCLE_ID", "lon": "LONGITUDE", "lat": "LATITUDE", "status": "LOCK_STATUS", "time": "DATA_TIME"}

def read_raw(path):
    """Read the raw log with pyarrow; BICYCLE_ID forced to string; date-only timestamps completed with 00:00:00."""
    import pyarrow as pa, pyarrow.csv as pcsv
    c = RAW_COLS
    t = pcsv.read_csv(str(path), convert_options=pcsv.ConvertOptions(include_columns=list(c.values()),
                      column_types={c["bike"]: pa.string(), c["status"]: pa.int8(), c["time"]: pa.string()}),
                      read_options=pcsv.ReadOptions(block_size=256 << 20)).to_pandas().rename(columns={v: k for k, v in c.items()})
    s = t["time"].astype(str); s = s.where(s.str.contains(" "), s + " 0:00:00")
    try:    t["time"] = pd.to_datetime(s, format="%Y/%m/%d %H:%M:%S")
    except ValueError: t["time"] = pd.to_datetime(s, format="mixed")
    n0 = len(t)
    t = t[t["lon"].between(BBOX[0], BBOX[1]) & t["lat"].between(BBOX[2], BBOX[3]) & t["time"].between(DATE_START, pd.Timestamp(DATE_END) + pd.Timedelta(days=1))]
    print(f"read {n0:,} rows -> after bounding-box and date filter {len(t):,} rows | {t['time'].min()} ~ {t['time'].max()}")
    return t.reset_index(drop=True)

def clean_events(t):
    """Sort by bike and time; drop mirror rows and jitter <= MIN_SEGMENT_SECONDS; EVENT_DEF='transition' keeps lock-state changes only."""
    t = t.sort_values(["bike", "time"]).reset_index(drop=True)
    b = t["bike"].to_numpy(); tm = t["time"].to_numpy(); s_ = t["status"].to_numpy()
    same = b[1:] == b[:-1]; dt = (tm[1:] - tm[:-1]) / np.timedelta64(1, "s")
    same_loc = np.isclose(t["lon"].to_numpy()[1:], t["lon"].to_numpy()[:-1]) & np.isclose(t["lat"].to_numpy()[1:], t["lat"].to_numpy()[:-1])
    rev = s_[1:] == 1 - s_[:-1]
    mirror = np.r_[same & (dt == 1) & same_loc & rev, False]
    short  = np.r_[same & (dt <= MIN_SEGMENT_SECONDS) & ~(dt == 1), False]
    t = t[~mirror & ~short].reset_index(drop=True); n_clean = len(t)
    if EVENT_DEF == "transition":
        b = t["bike"].to_numpy(); s_ = t["status"].to_numpy()
        t = t[np.r_[True, (b[1:] != b[:-1]) | (s_[1:] != s_[:-1])]].reset_index(drop=True)
    print(f"dropped mirror {int(mirror.sum()):,} rows, jitter {int(short.sum()):,} rows -> {n_clean:,} records; event definition {EVENT_DEF} -> {len(t):,} events "
          f"(lock {int((t['status']==1).sum()):,} / unlock {int((t['status']==0).sum()):,}), {t['bike'].nunique():,} bikes")
    return t[["lon", "lat", "status", "time"]]

T0 = time.time()
raw = read_raw(RAW_BIKE); ev = clean_events(raw); del raw
daily = ev.groupby(ev["time"].dt.date).size(); print("events per day:", {str(k): int(v) for k, v in daily.items()}); print(f"{time.time()-T0:.0f} s")

In [ ]:
# Events -> nearest station within 1 km
P = 1 if DIST_METRIC == "manhattan" else 2
ev_xy = to_xy_m(ev["lon"], ev["lat"])
d_near, i_near = tree.query(ev_xy, k=1, p=P)

ref_rows = []
for r in REF_RADII:
    ok = d_near <= r; h = ev["time"].to_numpy()[ok].astype("datetime64[h]")
    per_sh = pd.Series(1, index=pd.MultiIndex.from_arrays([i_near[ok], h])).groupby(level=[0, 1]).sum()
    n_hours = len(np.unique(h)); ref_rows.append({"radius_m": r, "matched_%": round(100 * ok.mean(), 1), "events": int(ok.sum()), "mean_per_station_hour": round(per_sh.sum() / (N * n_hours), 2)})
ref = pd.DataFrame(ref_rows); print("nearest-station matching (all hours, all days):\n" + ref.to_string(index=False))

if ASSIGN == "nearest":
    ok = d_near <= CATCHMENT_M
    events = pd.DataFrame({"s": i_near[ok], "time": ev["time"].to_numpy()[ok], "lock": ev["status"].to_numpy()[ok].astype(np.int8), "dist_m": d_near[ok].round(1)})
else:
    hits = tree.query_ball_point(ev_xy, r=CATCHMENT_M, p=P)
    e_idx = np.repeat(np.arange(len(hits)), [len(h) for h in hits]); s_idx = np.concatenate(hits).astype(int) if len(e_idx) else np.array([], int)
    dist = np.abs(ev_xy[e_idx] - st_xy[s_idx]).sum(1) if P == 1 else np.sqrt(((ev_xy[e_idx] - st_xy[s_idx]) ** 2).sum(1))
    events = pd.DataFrame({"s": s_idx, "time": ev["time"].to_numpy()[e_idx], "lock": ev["status"].to_numpy()[e_idx].astype(np.int8), "dist_m": dist.round(1)})
events = events.sort_values(["s", "time"]).reset_index(drop=True)
events.to_parquet(OUT_DIR / f"events_{CATCHMENT_M}m_{ASSIGN}.parquet")
print(f"\n{ASSIGN} / {CATCHMENT_M} m: {len(events):,} events assigned to {events['s'].nunique()} stations | lock (ride_in) {int((events.lock==1).sum()):,} | unlock (ride_out) {int((events.lock==0).sum()):,} | median distance {events['dist_m'].median():.0f} m")

In [ ]:
# 5-min event counts for Section 11
ev5 = events.assign(t5=events["time"].dt.floor("5min")).groupby(["s", "t5", "lock"]).size().rename("n").reset_index()
ev5.to_parquet(OUT_DIR / "events_5min_nearest.parquet")

## 11. Statistical pipeline on the 1-km panel vs published scores (Table 10)

In [ ]:
# Configuration and published scores (Li et al., 2026, Table 3)
import re, json, math, time, warnings, itertools, glob
from pathlib import Path
import numpy as np, pandas as pd
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
from scipy.stats import nbinom
from scipy.spatial import cKDTree
warnings.filterwarnings("ignore")

PANEL_VARIANT = "euclidean_all"
PANEL_1KM  = PANEL_1KM_DIR / PANEL_VARIANT
STATION_FILE = PANEL_1KM / "stations.csv"
CACHE_5MIN   = PANEL_1KM / "events_5min_nearest.parquet"
OUT_DIR  = STATS_1KM_DIR / PANEL_VARIANT; OUT_DIR.mkdir(parents=True, exist_ok=True)

CATCHMENT_M  = 1000
RES_LIST     = [15, 30, 60]
FULL_DAY     = True
SERVICE_HOURS = ("05:30", "23:30")
DATE_START, DATE_END = "2022-12-12", "2023-01-01"
TEST_START   = "2022-12-29"
VAL_DAYS     = 2

SHORT_HOURS  = 3
FLOOR        = 0.01
GROUP_EDGES_PER_HOUR = (2.0, 10.0)
PROF_GRID = {"half_life": [1, 2, 3, 5, np.inf], "dshrink": [0, 2, 8], "k": [1, 3, 10, 30], "ratio_prior": [0, 5, 20, 50], "beta": [0.0, 0.25, 0.5, 0.75]}
STAT_MODELS = ["HA", "hist_static", "seasonal_naive", "persistence", "profile_raw", "profile_eb", "profile_eb_ratio"]

LI = pd.DataFrame([
    # res, model, unlock_MAE, unlock_RMSE, unlock_WMAPE, lock_MAE, lock_RMSE, lock_WMAPE
    (15, "HA",       3.807,  5.917, 0.504, 3.692,  5.725, 0.489), (15, "ARIMA",   3.884,  6.698, 0.585, 3.882,  6.092, 0.491),
    (15, "XGBoost",  3.401,  5.083, 0.450, 3.476,  5.090, 0.440), (15, "LightGBM",3.727,  5.475, 0.493, 3.245,  5.658, 0.430),
    (15, "LSTM",     3.730,  5.085, 0.494, 3.872,  5.391, 0.490), (15, "GRU",     3.588,  5.081, 0.475, 3.595,  5.596, 0.469),
    (15, "GCLSTM",   2.592,  4.487, 0.341, 2.999,  4.918, 0.378), (15, "CoGCLSTM",2.597,  4.446, 0.342, 2.793,  4.775, 0.352),
    (30, "HA",       7.074, 11.245, 0.468, 6.920, 10.735, 0.458), (30, "ARIMA",   7.993, 11.659, 0.540, 7.275, 11.487, 0.461),
    (30, "XGBoost",  6.225, 10.329, 0.412, 6.267, 10.256, 0.397), (30, "LightGBM",6.640, 11.921, 0.439, 6.161, 10.584, 0.453),
    (30, "LSTM",     5.991, 10.568, 0.392, 5.991, 10.568, 0.392), (30, "GRU",     6.802, 10.225, 0.450, 6.860, 10.987, 0.496),
    (30, "GCLSTM",   4.785,  8.556, 0.313, 5.116,  8.920, 0.320), (30, "CoGCLSTM",4.662,  8.221, 0.305, 4.904,  8.725, 0.307),
    (60, "HA",      13.345, 21.606, 0.442,13.761, 21.127, 0.423), (60, "ARIMA",  13.853, 21.703, 0.450,13.489, 21.358, 0.427),
    (60, "XGBoost", 11.793, 18.765, 0.390,12.700, 18.929, 0.402), (60, "LightGBM",12.122,22.313, 0.401,11.794, 20.538, 0.391),
    (60, "LSTM",    13.390, 19.679, 0.443,13.433, 20.529, 0.489), (60, "GRU",    14.582, 19.927, 0.482,14.379, 19.244, 0.404),
    (60, "GCLSTM",  10.204, 17.620, 0.329,11.935, 20.158, 0.369), (60, "CoGCLSTM",9.531, 16.809, 0.308,10.383, 17.845, 0.321),
], columns=["res", "model", "unlock_MAE", "unlock_RMSE", "unlock_WMAPE", "lock_MAE", "lock_RMSE", "lock_WMAPE"])
LI["unlock_mean_implied"] = LI["unlock_MAE"] / LI["unlock_WMAPE"]; LI["lock_mean_implied"] = LI["lock_MAE"] / LI["lock_WMAPE"]
print("Mean flow per station and step implied by Li et al. (MAE / WMAPE):")
print(LI.groupby("res")[["unlock_mean_implied", "lock_mean_implied"]].median().round(1).to_string())

In [ ]:
# Read station table and 5-min counts
import pyarrow.parquet as pq
st = pd.read_csv(STATION_FILE); N = len(st); print(f"{N} stations ({STATION_FILE.name})", flush=True)
T0 = time.time(); tbl = pq.read_table(CACHE_5MIN, columns=["s", "t5", "lock", "n"]); print(f"read {CACHE_5MIN.name}: {tbl.num_rows:,} rows, {time.time()-T0:.1f} s", flush=True)
ev5 = pd.DataFrame({"s": tbl.column("s").to_numpy().astype(np.int32),
                    "t5": pd.to_datetime(tbl.column("t5").to_numpy()).tz_localize(None),
                    "lock": tbl.column("lock").to_numpy().astype(np.int8),
                    "n": tbl.column("n").to_numpy().astype(np.int64)}); del tbl
t5 = ev5["t5"].to_numpy(); keep = (t5 >= np.datetime64(DATE_START)) & (t5 < np.datetime64(DATE_END) + np.timedelta64(1, "D"))
ev5 = ev5[keep].reset_index(drop=True)
assert int(ev5["s"].max()) < N, "station index out of range: stations.csv and events_5min_nearest.parquet must come from the same run"
lk = ev5["lock"].to_numpy(); nn = ev5["n"].to_numpy()
print(f"5-min event counts: {len(ev5):,} rows | lock (ride_in) {int(nn[lk == 1].sum()):,} | unlock (ride_out) {int(nn[lk == 0].sum()):,} | dates {ev5['t5'].min().date()} ~ {ev5['t5'].max().date()}")

In [ ]:
# Panels at 15/30/60 min; train / validation / test days
def build_panel(res_min):
    e = ev5.copy(); e["t"] = e["t5"].dt.floor(f"{res_min}min")
    days = pd.date_range(e["t"].min().normalize(), e["t"].max().normalize(), freq="D")
    if FULL_DAY: slots = pd.timedelta_range("0min", "23h59min", freq=f"{res_min}min")
    else:
        h0, h1 = pd.to_timedelta(SERVICE_HOURS[0] + ":00"), pd.to_timedelta(SERVICE_HOURS[1] + ":00"); slots = pd.timedelta_range("0min", "23h59min", freq=f"{res_min}min"); slots = slots[(slots >= h0) & (slots <= h1)]
    SPD = len(slots); times = pd.DatetimeIndex([d + s for d in days for s in slots]); T = len(times)
    ti = pd.Series(np.arange(T), index=times); e = e[e["t"].isin(ti.index)]
    Y = np.zeros((T, N, 2)); np.add.at(Y, (ti[e["t"]].to_numpy(), e["s"].to_numpy(), 1 - e["lock"].to_numpy()), e["n"].to_numpy())
    day_id = np.repeat(np.arange(len(days)), SPD); slot = np.tile(np.arange(SPD), len(days)); is_wk = np.asarray(days.weekday >= 5).astype(int)
    test_day0 = int(np.searchsorted(days, pd.Timestamp(TEST_START))); n_days = len(days)
    tv_days = np.arange(0, test_day0); tr_days = tv_days[:-VAL_DAYS]; va_days = tv_days[-VAL_DAYS:]; te_days = np.arange(test_day0, n_days)
    return dict(res=res_min, Y=Y, times=times, days=days, day_id=day_id, slot=slot, SPD=SPD, T=T, n_days=n_days, is_wk=is_wk,
                tr_days=tr_days, va_days=va_days, te_days=te_days, steps_short=max(1, int(round(SHORT_HOURS * 60 / res_min))))

panels = {r: build_panel(r) for r in RES_LIST}
for r, P in panels.items():
    m = P["Y"].mean((0, 1)); te = np.isin(P["day_id"], P["te_days"])
    print(f"[{r:>2} min] {P['n_days']} days x {P['SPD']} steps x {N} stations | mean per station-step lock {m[0]:.2f} / unlock {m[1]:.2f} | zero share {100*(P['Y'][:,:,1]==0).mean():.0f}% | "
          f"train {P['days'][P['tr_days'][0]].date()}–{P['days'][P['tr_days'][-1]].date()} validation {len(P['va_days'])} days, test {P['days'][P['te_days'][0]].date()}–{P['days'][P['te_days'][-1]].date()} ({te.sum()} steps)")
print(f"\nCompare with the mean per station and step implied by Li et al. (about 30 / 32 at 60 min); panel variant {PANEL_VARIANT}.")

In [ ]:
# Functions: statistical pipeline and evaluation
def build_profiles(P, hl, ds):
    """(a) Causal profile: day d uses earlier days only, weights 0.5^((d-i-1)/hl); day-type profile shrunk towards all days. Returns prof (D,H,N,2), W_eff (D,)."""
    nd, SPD = P["n_days"], P["SPD"]; Yd = P["Y"].reshape(nd, SPD, N, 2); i = np.arange(nd); lag = i[:, None] - i[None, :] - 1
    wgt = np.where(lag >= 0, (0.5 ** (np.maximum(lag, 0) / hl)) if np.isfinite(hl) else 1.0, 0.0); Wdt = wgt * (P["is_wk"][:, None] == P["is_wk"][None, :])
    raw_all = np.einsum("di,ihsc->dhsc", wgt, Yd) / np.maximum(wgt.sum(1), 1e-12)[:, None, None, None]
    raw_dt = np.einsum("di,ihsc->dhsc", Wdt, Yd) / np.maximum(Wdt.sum(1), 1e-12)[:, None, None, None]
    Wd = Wdt.sum(1); den = (Wd + ds)[:, None, None, None]
    prof = np.where(den > 0, (Wd[:, None, None, None] * raw_dt + ds * raw_all) / np.maximum(den, 1e-12), raw_all)
    return prof, np.minimum(Wd + ds, wgt.sum(1))

def eb_shrink(prof, W_eff, k):
    """(b) Empirical-Bayes shrinkage: prior mean m = station level x network hourly shape, base = (k + W*prof)/(k/m + W)."""
    level = prof.mean(1, keepdims=True); net = prof.mean(2, keepdims=True); shape = net / np.maximum(net.mean(1, keepdims=True), 1e-9)
    m = np.maximum(level * shape, 1e-6); W = W_eff[:, None, None, None]; return (k + W * prof) / (k / m + W)

def _winsum(a, L):
    c = np.cumsum(np.concatenate([np.zeros((1,) + a.shape[1:]), a], 0), 0); t = np.arange(len(a)); out = np.zeros_like(a); ok = t - L >= 0
    out[ok] = c[t[ok]] - c[t[ok] - L]; return out

def level_ratio(P, B, prior, beta):
    """(c) Recent-level ratio: previous service day and previous SHORT_HOURS hours, combined geometrically."""
    Y, SPD, S = P["Y"], P["SPD"], P["steps_short"]; rd = (_winsum(Y, SPD) + prior) / (_winsum(B, SPD) + prior)
    ps = prior * S / SPD; rs = (_winsum(Y, S) + ps) / (_winsum(B, S) + ps); return rd ** (1 - beta) * rs ** beta

def pois_dev(y, mu):
    mu = np.maximum(mu, FLOOR); return float(np.mean(2 * (np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0.0) - (y - mu))))

def fit_phi(y, mu):
    y = y.ravel(); mu = np.maximum(mu.ravel(), FLOOR)
    def nll(lp):
        phi = math.exp(lp); return -np.mean(gammaln(y + phi) - gammaln(phi) - gammaln(y + 1) + phi * np.log(phi / (phi + mu)) + y * np.log(mu / (phi + mu)))
    return math.exp(minimize_scalar(nll, bounds=(-3, 9), method="bounded").x)

def targets(P, days, warm=1):
    """Target times in the given days with at least `warm` full days of history."""
    return np.where(np.isin(P["day_id"], days) & (P["day_id"] >= warm))[0]

def ha_pred(P, hd):
    """Historical average of Li et al.: mean of the same time slot over training days."""
    Yd = P["Y"].reshape(P["n_days"], P["SPD"], N, 2); H = Yd[hd].mean(0); return H[P["slot"]]

def hist_static_pred(P, hd):
    Yd = P["Y"].reshape(P["n_days"], P["SPD"], N, 2); allm = Yd[hd].mean(0); H = np.zeros((2, P["SPD"], N, 2))
    for typ in (0, 1):
        dd = hd[P["is_wk"][hd] == typ]; H[typ] = Yd[dd].mean(0) if len(dd) else allm
    return H[P["is_wk"][P["day_id"]], P["slot"]]

def poisson_mae_floor(mu, kmax=400):
    """E|Y - mu| for Y ~ Poisson(mu): the Poisson reference error."""
    mu = np.maximum(np.asarray(mu, float), 1e-9); k = np.arange(kmax + 1)[:, None]; mu_f = mu.ravel()[None, :]
    pmf = np.exp(k * np.log(mu_f) - mu_f - gammaln(k + 1)); return (np.abs(k - mu_f) * pmf).sum(0).reshape(mu.shape)

def eval_block(y, p, phi=None):
    out = {}; p = np.maximum(p, 0)
    for j, tgt in enumerate(("lock", "unlock")):
        yt, pt = y[:, :, j].ravel(), p[:, :, j].ravel(); e = yt - pt
        out[f"{tgt}_MAE"] = np.abs(e).mean(); out[f"{tgt}_RMSE"] = np.sqrt((e ** 2).mean()); out[f"{tgt}_WMAPE"] = np.abs(e).sum() / max(yt.sum(), 1e-9)
        out[f"{tgt}_bias_ratio"] = pt.sum() / max(yt.sum(), 1e-9); out[f"{tgt}_R2"] = 1 - (e ** 2).sum() / ((yt - yt.mean()) ** 2).sum()
        if phi is not None:
            mu = np.maximum(pt, FLOOR); ph = np.broadcast_to(phi[:, j], y[:, :, j].shape).ravel(); pp = ph / (ph + mu)
            out[f"{tgt}_NLL"] = -nbinom.logpmf(yt, ph, pp).mean(); out[f"{tgt}_cov90"] = ((yt >= nbinom.ppf(0.05, ph, pp)) & (yt <= nbinom.ppf(0.95, ph, pp))).mean()
    return out

In [ ]:
# Parameter selection, test evaluation, comparison tables
rows, preds_all, params_all, curves = [], {}, {}, {}
T0 = time.time()
for r, P in panels.items():
    tv_t = targets(P, np.r_[P["tr_days"], P["va_days"]], warm=2); te_t = targets(P, P["te_days"], warm=1); hd = P["tr_days"]
    lvl = P["Y"][np.isin(P["day_id"], P["tr_days"])].mean(0)
    best, grid = None, []
    for hl, ds in itertools.product(PROF_GRID["half_life"], PROF_GRID["dshrink"]):
        prof, W = build_profiles(P, hl, ds)
        for k in PROF_GRID["k"]:
            B = eb_shrink(prof, W, k).reshape(P["T"], N, 2)
            for pr, be in itertools.product(PROF_GRID["ratio_prior"], PROF_GRID["beta"]):
                dev = pois_dev(P["Y"][tv_t], (B * level_ratio(P, B, pr, be))[tv_t]); cand = dict(half_life=hl, dshrink=ds, k=k, ratio_prior=pr, beta=be)
                grid.append({**cand, "dev": dev})
                if best is None or dev < best[0]: best = (dev, cand)
    bp = best[1]; pd.DataFrame(grid).to_csv(OUT_DIR / f"profile_grid_{r}min.csv", index=False)
    prof, W = build_profiles(P, bp["half_life"], bp["dshrink"]); Praw = prof.reshape(P["T"], N, 2)
    B = eb_shrink(prof, W, bp["k"]).reshape(P["T"], N, 2); MU = B * level_ratio(P, B, bp["ratio_prior"], bp["beta"])
    preds = {"HA": ha_pred(P, hd), "hist_static": hist_static_pred(P, hd), "seasonal_naive": np.r_[np.zeros((P["SPD"], N, 2)), P["Y"][:-P["SPD"]]],
             "persistence": np.r_[np.zeros((1, N, 2)), P["Y"][:-1]], "profile_raw": Praw, "profile_eb": B, "profile_eb_ratio": MU}
    edges = np.array(GROUP_EDGES_PER_HOUR) * r / 60; grp = np.digitize(lvl, edges); phi = np.ones((N, 2))
    for g in range(3):
        mk = grp == g
        if mk.any(): phi[mk] = fit_phi(P["Y"][tv_t][:, mk], MU[tv_t][:, mk])
    y_te = P["Y"][te_t]; floor = poisson_mae_floor(np.maximum(MU[te_t], FLOOR))
    for m in STAT_MODELS:
        p_te = preds[m][te_t]; rr = eval_block(y_te, p_te, phi if m == "profile_eb_ratio" else None)
        rr.update({"res": r, "model": m, "n_test_steps": len(te_t), "lock_mean": y_te[:, :, 0].mean(), "unlock_mean": y_te[:, :, 1].mean(),
                   "lock_MAE_floor": floor[:, :, 0].mean(), "unlock_MAE_floor": floor[:, :, 1].mean(),
                   "lock_WMAPE_floor": floor[:, :, 0].sum() / y_te[:, :, 0].sum(), "unlock_WMAPE_floor": floor[:, :, 1].sum() / y_te[:, :, 1].sum()})
        for j, tgt in enumerate(("lock", "unlock")):
            sm = np.abs(y_te[:, :, j] - np.maximum(p_te[:, :, j], 0)).mean(0)
            for g, gname in enumerate(("small", "mid", "large")): rr[f"{tgt}_MAE_{gname}"] = sm[grp[:, j] == g].mean() if (grp[:, j] == g).any() else np.nan
        rows.append(rr)
    preds_all[r] = {m: preds[m][te_t] for m in STAT_MODELS}; preds_all[r]["y"] = y_te; preds_all[r]["te_t"] = te_t
    phi_g = {g: float(np.unique(phi[grp == g])[0]) for g in range(3) if (grp == g).any()}; params_all[r] = {**bp, "phi_by_group": phi_g, "dev_trainval": best[0]}
    curves[r] = {"times": P["times"][te_t], "y_tot": y_te.sum(1), "pred_tot": {m: np.maximum(preds[m][te_t], 0).sum(1) for m in ("HA", "profile_eb_ratio")}}
    np.savez_compressed(OUT_DIR / f"preds_{r}min.npz", te_t=te_t, y=y_te, lvl=lvl, phi=phi, **{m: preds[m][te_t] for m in STAT_MODELS})
    print(f"[{r:>2} min] parameters {bp} | φ {phi_g} | {(time.time()-T0)/60:.1f} min")

res_df = pd.DataFrame(rows); res_df.to_csv(OUT_DIR / "stats_1km_metrics.csv", index=False)
json.dump(params_all, open(OUT_DIR / "stats_1km_params.json", "w"), indent=2, default=float)

show = ["unlock_MAE", "unlock_RMSE", "unlock_WMAPE", "lock_MAE", "lock_RMSE", "lock_WMAPE"]
for r in RES_LIST:
    ours = res_df[res_df.res == r].set_index("model")[show + ["unlock_mean", "lock_mean", "unlock_WMAPE_floor", "lock_WMAPE_floor", "unlock_bias_ratio"]]
    li = LI[LI.res == r].set_index("model")[show]
    print(f"\n===== {r} min | 1-km nearest station | test {panels[r]['days'][panels[r]['te_days'][0]].date()}–{panels[r]['days'][panels[r]['te_days'][-1]].date()} =====")
    print(f"mean per station-step: ours unlock {ours['unlock_mean'].iloc[0]:.2f} / lock {ours['lock_mean'].iloc[0]:.2f}; implied by Li et al. {LI[LI.res==r]['unlock_mean_implied'].median():.1f} / {LI[LI.res==r]['lock_mean_implied'].median():.1f}")
    print(f"Poisson reference WMAPE: unlock {ours['unlock_WMAPE_floor'].iloc[0]:.3f} / lock {ours['lock_WMAPE_floor'].iloc[0]:.3f}")
    both = pd.concat([ours[show].assign(source="ours (statistical)"), li.assign(source="Li et al. 2026")]); both = both[["source"] + show]
    print(both.round(3).to_string())
    both.to_csv(OUT_DIR / f"compare_li_{r}min.csv")

summ = []
for r in RES_LIST:
    o = res_df[res_df.res == r].set_index("model"); li = LI[LI.res == r].set_index("model")
    for tgt in ("unlock", "lock"):
        summ.append({"res": r, "target": tgt, "mean_per_station_step": round(o[f"{tgt}_mean"].iloc[0], 2), "floor": round(o[f"{tgt}_WMAPE_floor"].iloc[0], 3),
                     "ours_HA": round(o.loc["HA", f"{tgt}_WMAPE"], 3), "ours_profile_eb_ratio": round(o.loc["profile_eb_ratio", f"{tgt}_WMAPE"], 3),
                     "ours_gap_to_floor": round(o.loc["profile_eb_ratio", f"{tgt}_WMAPE"] - o[f"{tgt}_WMAPE_floor"].iloc[0], 3),
                     "Li_HA": li.loc["HA", f"{tgt}_WMAPE"], "Li_LSTM": li.loc["LSTM", f"{tgt}_WMAPE"], "Li_GCLSTM": li.loc["GCLSTM", f"{tgt}_WMAPE"], "Li_CoGCLSTM": li.loc["CoGCLSTM", f"{tgt}_WMAPE"],
                     "ours/CoGCLSTM": round(o.loc["profile_eb_ratio", f"{tgt}_WMAPE"] / li.loc["CoGCLSTM", f"{tgt}_WMAPE"], 3), "ours_bias_ratio": round(o.loc["profile_eb_ratio", f"{tgt}_bias_ratio"], 3)})
summ = pd.DataFrame(summ); summ.to_csv(OUT_DIR / "summary_vs_li.csv", index=False)
print("\n===== Summary: WMAPE (ours/CoGCLSTM < 1 = statistical pipeline better) =====\n" + summ.to_string(index=False))

## 12. Figures

### 12a. Station-level results, Figure 7 (reads the outputs of Section 7)

In [ ]:
# Figure: fig_station_level
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon

ST_DIR = FORECAST_DIR / "station"
summ = pd.read_csv(ST_DIR / "summary.csv"); summ = summ[summ["res"] == "1h"]
runs = pd.read_csv(ST_DIR / "results_all.csv"); runs = runs[(runs["res"] == "1h") & (runs["attention"].astype(str).str.lower() != "true")]

models = ["LSTM", "ST-GCN", "MBC-GCN"]
def cell_of(model, features):
    r = summ[(summ["model"] == model) & (summ["features"] == features) & (summ["attention"].astype(str).str.lower() != "true")].iloc[0]
    return float(r["ride_out_MAE_mean"]), float(r["ride_out_MAE_std"])
ar    = {m: cell_of(m, "ar") for m in models}
semi  = {m: cell_of(m, "semi_ar") for m in models}
naive = [(lab, *cell_of(name, "-")) for lab, name in (("hist.\nmean", "hist_mean"), ("seasonal\nnaive", "seasonal_naive"), ("persist-\nence", "persistence"))]
gain = {}
key = ["fold", "seed", "model"]
d_ar = runs[runs["features"] == "ar"].set_index(key)["ride_out_MAE"]; d_sa = runs[runs["features"] == "semi_ar"].set_index(key)["ride_out_MAE"]
for m in models:
    diff = (d_ar - d_sa).xs(m, level="model").dropna(); pct = 100 * diff / d_ar.xs(m, level="model").loc[diff.index]
    gain[m] = (pct.mean(), pct.std(), wilcoxon(diff, alternative="greater").pvalue)

plt.rcParams.update({"font.family": ["Arial", "Liberation Sans", "DejaVu Sans"], "font.size": 12,
                     "axes.edgecolor": "#5f6b7a", "axes.labelcolor": "#1f2937", "xtick.color": "#5f6b7a", "ytick.color": "#5f6b7a"})
C_AR, C_SEMI, C_NAIVE, C_GAIN, C_ERR = "#9fb3d9", "#2f5597", "#b8bfca", "#1f3a5f", "#5f6b7a"
fig, (a, b) = plt.subplots(1, 2, figsize=(12.6, 4.3), gridspec_kw={"width_ratios": [1.45, 1]})
ek = dict(ecolor=C_ERR, elinewidth=1.2, capsize=4, capthick=1.2)

# (a) departure MAE
w = 0.36
for i, m in enumerate(models):
    a.bar(i - w/2, ar[m][0],   w, yerr=ar[m][1],   color=C_AR,   error_kw=ek, label="AR (bike only)" if i == 0 else None)
    a.bar(i + w/2, semi[m][0], w, yerr=semi[m][1], color=C_SEMI, error_kw=ek, label="Semi-AR (bike + metro)" if i == 0 else None)
xn = [3.75, 4.75, 5.75]
for j, (x, (_, v, s)) in enumerate(zip(xn, naive)):
    a.bar(x, v, 0.45, yerr=s, color=C_NAIVE, error_kw=ek, label="naive references" if j == 0 else None)
    a.text(x, v + s + 0.015, f"{v:.3f}", ha="center", va="bottom", fontsize=10.5, color="#1f2937")
a.axhline(naive[0][1], ls="--", lw=1.2, color=C_ERR, zorder=0)
a.text(3.05, naive[0][1] + 0.012, "hist_mean", ha="center", va="bottom", fontsize=10.5, color=C_ERR)
a.set_xticks([0, 1, 2] + xn); a.set_xticklabels(models + [n for n, _, _ in naive])
a.set_ylim(0.85, 1.62); a.set_yticks(np.arange(0.9, 1.61, 0.1))
a.set_ylabel("ride_out MAE per station–hour")
a.set_title("(a) Station-level 1-h forecasts, 342 stations (3 folds × 3 seeds)", fontsize=13, color="#1f2937")
a.grid(axis="y", color="#e5e7eb", lw=0.8); a.set_axisbelow(True)
a.legend(frameon=False, loc="upper left", fontsize=11, borderaxespad=0.2)

# (b) reduction in MAE from metro inputs
for i, m in enumerate(models):
    g, s, p = gain[m]; y = len(models) - 1 - i
    b.barh(y, g, 0.5, xerr=s, color=C_GAIN, error_kw=ek)
    b.text(g + s + 0.25, y, f"+{g:.1f} %  (p = {p:.3f})", va="center", fontsize=11, color="#1f2937")
b.axvline(0, color=C_ERR, lw=1.0)
b.set_yticks(range(len(models))); b.set_yticklabels(models[::-1], fontsize=12)
b.set_xlim(-0.5, 7.5); b.set_xticks([0, 2, 4, 6]); b.set_ylim(-0.5, 2.5)
b.set_xlabel("ride_out MAE reduction, AR → Semi-AR (%)")
b.set_title("(b) Metro-information gain (Wilcoxon over 9 matched runs)", fontsize=13, color="#1f2937")
b.grid(axis="x", color="#e5e7eb", lw=0.8); b.set_axisbelow(True)
for ax in (a, b): ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(w_pad=3)
plt.savefig(FIG_DIR / "fig_station_level.png", dpi=300, bbox_inches="tight"); plt.savefig(FIG_DIR / "fig_station_level.pdf", bbox_inches="tight"); plt.show()

### 12b. Network-total forecasts, Figures 8 and 9 (reads the outputs of Section 9)

In [ ]:
# Paths and plot style
from pathlib import Path
import pickle, logging, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

DIR_06    = FORECAST_DIR / "network_total"
PANEL_CSV = PANEL_DIR / "panel_1h_200m.csv"
DIR_1KM   = STATS_1KM_DIR / "euclidean_all"
RES = "1h"; SERVICE = ("05:30", "23:30"); RES_1KM = [15, 30, 60]

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
                     "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.7, "xtick.major.width": 0.7, "ytick.major.width": 0.7,
                     "figure.dpi": 200, "pdf.fonttype": 42, "savefig.facecolor": "white"})
COL   = {"naive": "#9E9E9E", "statistical": "#204074", "sarima": "#6B4D93", "ridge": "#407FBF", "hgb": "#2B9D8F", "deep": "#E07C3A"}
NAME  = {"hist_mean": "Historical mean", "seasonal_naive": "Seasonal naive", "profile_ratio": "Statistical pipeline", "sarima_semi_ar": "SARIMAX (Semi-AR)",
         "ridge_semi_ar": "Ridge (Semi-AR)", "hgb_semi_ar": "Gradient boosting (Semi-AR)", "LSTM semi_ar": "LSTM (Semi-AR)",
         "ST-GCN semi_ar": "ST-GCN (Semi-AR)", "MBC-GCN semi_ar": "MBC-GCN (Semi-AR)"}
STYLE = {"hist_mean":      dict(ls=(0, (1.2, 1.2)), lw=0.9),
         "seasonal_naive": dict(ls=(0, (4, 1.6)),   lw=0.9),
         "profile_ratio":  dict(ls=(0, (5, 1.5)),   lw=1.4, marker="o", ms=2.4, markevery=(0, 2)),
         "sarima_semi_ar": dict(ls=(0, (3, 1, 1, 1)), lw=1.0, marker="s", ms=2.2, markevery=(1, 2)),
         "ridge_semi_ar":  dict(ls="-",             lw=0.8, marker="^", ms=2.4, markevery=(0, 2)),
         "hgb_semi_ar":    dict(ls="-",             lw=0.8, marker="v", ms=2.4, markevery=(1, 2)),
         "LSTM semi_ar":   dict(ls="-",             lw=1.2, marker="D", ms=2.0, markevery=(0, 2))}
FAM_ORDER = ["naive", "statistical", "sarima", "ridge", "hgb", "deep"]
PANEL = "abcdefgh"
def style(name): return {**STYLE.get(name, dict(ls="-", lw=0.9)), "mew": 0}
def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", pad_inches=0.02)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight", pad_inches=0.02)
    print("saved", FIG_DIR / f"{name}.pdf")

In [ ]:
# Read network-total results
S = pickle.load(open(DIR_06 / "session_results.pkl", "rb"))
summary, curves = S["summary"], S["curves"]
tab = summary[RES]; tab = tab[~tab["label"].str.contains("attn")].reset_index(drop=True)
cv = {k[1]: v for k, v in curves.items() if k[0] == RES}; N_FOLDS = len(cv)

p = pd.read_csv(PANEL_CSV, usecols=["window"]); p["window"] = pd.to_datetime(p["window"]); h = p["window"].dt.time
p = p[(h >= pd.to_datetime(SERVICE[0]).time()) & (h <= pd.to_datetime(SERVICE[1]).time())]
w = pd.DatetimeIndex(np.sort(p["window"].unique())); dates = w.normalize(); udays = np.unique(dates); day_id = np.searchsorted(udays, dates)
hours = w.hour + w.minute / 60
assert len(w) >= max(v["tg"].max() for v in cv.values()) + 1, "panel time axis is shorter than the target index of the network-total results"

best = tab.loc[tab.groupby("family")["ride_out_MAE_mean"].idxmin()].set_index("family")["label"].to_dict()
show = ["hist_mean"] + [best[f] for f in FAM_ORDER if f in best and best[f] != "hist_mean"]
fam_of = tab.set_index("label")["family"].to_dict()
print("best model of each family:", show)

In [ ]:
# Figure: fig_total_curves
def plot_curves(models, fname):
    """Plot actual totals and the listed models over the test windows; returns fold MAEs."""
    fig, axes = plt.subplots(N_FOLDS, 2, figsize=(6.5, 1.95 * N_FOLDS + 0.55), sharey="col"); axes = np.atleast_2d(axes)
    rows = []; ymax = [0, 0]
    for f in range(N_FOLDS):
        v = cv[f + 1]; tg = v["tg"]; x = np.arange(len(tg)); dstart = np.r_[0, np.where(np.diff(day_id[tg]) > 0)[0] + 1]
        for j, tgt in enumerate(("ride_in", "ride_out")):
            ax = axes[f, j]
            for name in models:
                pr = v["preds"][name][:, j]; fam = fam_of[name]
                ax.plot(x, pr, color=COL[fam], label=NAME.get(name, name), **style(name)); ymax[j] = max(ymax[j], pr.max())
                rows.append((f + 1, tgt, NAME.get(name, name), np.abs(v["y"][:, j] - pr).mean()))
            ax.plot(x, v["y"][:, j], color="k", lw=1.6, label="Actual", zorder=5); ymax[j] = max(ymax[j], v["y"][:, j].max())
            for d_ in dstart[1:]: ax.axvline(d_ - 0.5, color="#BDBDBD", lw=0.7)
            ax.set_xticks(dstart); ax.set_xticklabels(dates[tg[dstart]].strftime("%d %b")); ax.xaxis.set_minor_locator(MultipleLocator(6))
            ax.tick_params(axis="x", which="minor", length=2); ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_xlim(-0.5, len(tg) - 0.5)
            ax.set_title(f"({PANEL[2*f+j]}) Fold {f+1}, {'arrivals (ride_in)' if tgt == 'ride_in' else 'departures (ride_out)'}", loc="left")
            if j == 0: ax.set_ylabel("Trips per hour")
    for j in range(2): axes[0, j].set_ylim(0, ymax[j] * 1.08)
    for ax in axes[-1]: ax.set_xlabel("Test day")
    hh, ll = axes[0, 0].get_legend_handles_labels(); order = [ll.index("Actual")] + [i for i, l in enumerate(ll) if l != "Actual"]
    fig.legend([hh[i] for i in order], [ll[i] for i in order], loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.005), handlelength=2.6, columnspacing=1.4)
    fig.tight_layout(rect=(0, 0.06, 1, 1)); save(fig, fname); plt.show()
    return rows

plot_curves(show, "fig_total_curves")

In [ ]:
# Figure: fig_total_hourly
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.6), sharey=True)
hr = np.concatenate([hours[v["tg"]] for v in cv.values()])
for j, tgt in enumerate(("ride_in", "ride_out")):
    yy = np.concatenate([v["y"][:, j] for v in cv.values()]); ax = axes[j]
    for name in show:
        e = np.abs(yy - np.concatenate([v["preds"][name][:, j] for v in cv.values()])); s = pd.Series(e).groupby(hr).mean()
        st = style(name); ax.plot(s.index, s.to_numpy(), marker="o", ms=2.3, mew=0, color=COL[fam_of[name]], label=NAME.get(name, name), ls=st["ls"], lw=st["lw"])
    ax.set_xlabel("Hour of day"); ax.set_title(f"({PANEL[j]}) {'Arrivals (ride_in)' if tgt == 'ride_in' else 'Departures (ride_out)'}", loc="left")
    ax.grid(axis="y", alpha=0.25, lw=0.6); ax.set_xticks(range(6, 24, 2)); ax.set_xlim(5.3, 23.7)
axes[0].set_ylabel("MAE (trips per hour)")
hh, ll = axes[0].get_legend_handles_labels()
fig.legend(hh, ll, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.01), handlelength=2.2, columnspacing=1.4)
fig.tight_layout(rect=(0, 0.14, 1, 1)); save(fig, "fig_total_hourly"); plt.show()

### 12c. WMAPE under the 1-km protocol, Figure 11 (reads the outputs of Section 11)

In [ ]:
# Read 1-km comparison
LI_ORDER = ["HA", "ARIMA", "XGBoost", "LightGBM", "LSTM", "GRU", "GCLSTM", "CoGCLSTM"]
ref = pd.read_csv(DIR_1KM / "summary_vs_li.csv").set_index(["res", "target"])["floor"]      # Poisson reference WMAPE
ours, li = {"unlock": {}, "lock": {}}, {"unlock": {}, "lock": {}}
for r in RES_1KM:
    d = pd.read_csv(DIR_1KM / f"compare_li_{r}min.csv").set_index("model")
    is_ours = d["source"].str.startswith("ours"); o, l = d[is_ours], d[~is_ours]
    for tgt in ("unlock", "lock"):
        ours[tgt][r] = dict(HA=float(o.loc["HA", f"{tgt}_WMAPE"]), pipe=float(o.loc["profile_eb_ratio", f"{tgt}_WMAPE"]),
                            floor=float(ref.loc[(r, tgt)]) if tgt == "unlock" else np.nan)   # reference line for departures only
        li[tgt][r] = {m: float(l.loc[m, f"{tgt}_WMAPE"]) for m in LI_ORDER if m in l.index}

In [ ]:
# Figures: fig_li_wmape_unlock, fig_li_wmape_lock
C_HA, C_PIPE, C_LI, C_COG, C_FLOOR = "#9E9E9E", "#1F3F73", "#D9A0A0", "#C23B3B", "#1F3F73"

def wmape_panel(ax, ours_t, li_t, ylabel):
    x0 = 0; centres = []; top = 0
    for r in RES_1KM:
        li_models = [m for m in LI_ORDER if m in li_t[r]]
        labs = ["HA*", "Pipeline*"] + li_models
        vals = [ours_t[r]["HA"], ours_t[r]["pipe"]] + [li_t[r][m] for m in li_models]
        cols = [C_HA, C_PIPE] + [C_COG if m == "CoGCLSTM" else C_LI for m in li_models]
        xs = x0 + np.arange(len(vals)); ax.bar(xs, vals, color=cols, width=0.78, zorder=2)
        if np.isfinite(ours_t[r]["floor"]): ax.hlines(ours_t[r]["floor"], xs[0] - 0.5, xs[-1] + 0.5, color=C_FLOOR, ls=(0, (4, 2)), lw=1.0, zorder=3)
        for x, v in zip(xs, vals): ax.text(x, v + 0.008, f"{v:.2f}", ha="center", va="bottom", fontsize=6.5, rotation=90, color="#1F2937")
        for x, lab in zip(xs, labs): ax.text(x, -0.012, lab, ha="center", va="top", fontsize=6.3, rotation=90, transform=ax.get_xaxis_transform(), color="#4B5563")
        centres.append(xs.mean()); top = max(top, max(vals)); x0 = xs[-1] + 2.2
    ax.set_xticks([]); ax.set_ylabel(ylabel); ax.set_ylim(0, top + 0.09); ax.grid(axis="y", alpha=0.25, lw=0.6, zorder=0); ax.set_xlim(-0.8, x0 - 2.2 + 0.8)
    ax.spines["bottom"].set_visible(False)
    for cx, r in zip(centres, RES_1KM): ax.text(cx, -0.34, f"{r} min", ha="center", va="top", fontsize=9, transform=ax.get_xaxis_transform())
handles = [Patch(color=C_HA, label="Historical average, this study"), Patch(color=C_PIPE, label="Statistical pipeline, this study"),
           Patch(color=C_LI, label="Published models"), Patch(color=C_COG, label="CoGCLSTM"),
           Line2D([0], [0], color=C_FLOOR, ls=(0, (4, 2)), lw=1.0, label="Poisson reference of our panel")]

for tgt, fname, ylabel in [("unlock", "fig_li_wmape_unlock", "WMAPE, departures (unlock)"), ("lock", "fig_li_wmape_lock", "WMAPE, arrivals (lock)")]:
    fig, ax = plt.subplots(figsize=(6.5, 3.5)); wmape_panel(ax, ours[tgt], li[tgt], ylabel)
    hs = handles if np.isfinite(ours[tgt][15]["floor"]) else handles[:4]
    ax.legend(handles=hs, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False, columnspacing=1.2, handlelength=1.4)
    fig.tight_layout(); save(fig, fname); plt.show()